# Piece Design Document — HexMagic

## 1. Piece Roster

### Summary Table

| Piece    | Role                  | Speed | Diet  | Capacity | Harvest | Attack | Defense |
|----------|-----------------------|-------|-------|----------|---------|--------|---------|
| **King** | Base / Activator      | 2     | 5.0   | 14.0     | 0.50    | ⭐     | ⭐      |
| **Queen**| Commander / Builder   | 3     | 27.9  | 59.0     | 0.48    | ⭐⭐⭐ | ⭐⭐⭐  |
| **Bishop**| Food Banker / Trucker| 3     | 1.9   | 178.7    | 0.84    | ⭐     | ⭐      |
| **Knight**| Scout / Raider       | 4     | 4.0   | 10.0     | 1.50    | ⭐⭐   | ⭐      |
| **Rook** | Fortress / Anchor     | 0–1   | 3.5   | 12.0     | 1.80    | ⭐     | ⭐⭐⭐  |
| **Pawn** | Harvester / Garrison  | 2     | 2.7   | 8.7      | 2.06    | ⭐     | ⭐      |

Values are from the Optuna-tuned FoodProfile (v5, trial 108, error 122.6).
Rook and Knight profiles are design targets, not yet Optuna-tuned.

---

## 2. Piece Roles — Detailed

### King
- **Purpose:** Victory condition and egg activator. Killing the enemy king wins the game.
- **Combat:** Nearly useless. Must be protected at all times.
- **Economy:** Low harvest, moderate reserves. Needs at least one pawn escort.
- **Strategic value:** Stays near home settlement. His presence "activates" queen eggs
  (see §6 Spawning). Moving the king forward is a desperate late-game gambit.

### Queen
- **Purpose:** Powerhouse commander. Best attacker, best defender, worst eater.
- **Combat:** Dominates her sight cone from high ground. Devastating on offense or defense.
- **Economy:** Massive caloric deficit. On the best terrain (tier 7), her net per turn is
  still −24.5. She CANNOT self-sustain on any terrain. This is the core design constraint —
  it forces meaningful escort composition.
- **Builder:** Can construct rooks (immobile fortresses) at a cost (see §6 Spawning).
- **Key stat:** 59 food / 27.9 diet = ~2.1 turns of unsupported travel. She's the bottleneck.

### Bishop
- **Purpose:** The food economy's central bank. Stores and redistributes surplus.
- **Combat:** Terrible. The most strategically important piece and the worst fighter.
  A knight's ideal raid target, which creates genuine escort decisions.
- **Economy:** 178.7 capacity = can fund ~6 full queen refills. Moderate harvest (0.84)
  means it slowly refills itself on tier 3+ terrain. 178.7 / 1.9 = ~94 turns solo endurance.
- **Trucker role:** Bishop truckers shuttle food from fertile waystations to the front line.
  Effective round-trip radius ≈ capacity / (2 × diet) ≈ 47 turns.

### Knight
- **Purpose:** Fast scout and raider. Finds waystation locations, picks off unguarded targets.
- **Combat:** Good attack, poor defense. Excels at hit-and-run. Gets slaughtered by a queen
  with high ground (vision cone + attack strength).
- **Economy:** Self-sustaining on tier 3+ terrain (1.5 × 3 − 4.0 = +0.5). Speed 4 means
  it covers ground fast but burns food proportionally.
- **Key tension:** Can destroy an unguarded pawn waystation and cut a supply line,
  but a defended waystation (rook + pawns) will shred it.

### Rook
- **Purpose:** Immobile (or nearly) fortress. Built by the queen, not recruited.
- **Combat:** Minimal attack but extremely high defense. A rook on high ground is
  a permanent no-go zone for knights and unescorted bishops.
- **Economy:** Low diet (3.5) with decent harvest (1.8). On tier 2+ terrain it's
  self-sustaining. On high-tier terrain it becomes a supply anchor — passively
  accumulating food for bishop truckers to collect.
- **Strategic role:** A waystation upgrade. Pawn garrison = temporary and vulnerable.
  Rook = permanent and hard to dislodge. Location matters economically — a rook on
  tier 5 is worth building, tier 1 is just a roadblock.

### Pawn
- **Purpose:** The workhorse. Harvests food, garrisons waystations, fights in numbers.
- **Combat:** Weak individually, dangerous in groups. Cannon fodder.
- **Economy:** Highest harvest strength (2.06). Break-even at tier 2 (2.06×2=4.12 > 2.7).
  At tier 4: net +5.54/turn. At tier 7: net +11.72/turn. The primary food producers.
- **Key math:** It takes ~5 pawns on tier 4 (or ~3 on tier 7) to feed one queen.

---

## 3. Food Economy

### Core Formula
```
gross_harvest = harvest_strength × food_tier
net_per_turn  = gross_harvest − diet
turns_of_reserve = food / diet
```

### The Queen Deficit Principle
The queen is ALWAYS in caloric deficit. This is not a bug — it's THE design constraint.
On tier 7 (best possible): 0.48 × 7 − 27.9 = −24.54/turn.
This forces every strategic decision to revolve around "how do I keep the queen fed?"

### Starvation
```
shortfall = max(0, diet − food)
damage = shortfall × STARVE_MULT (currently 5)
```
A queen with zero food takes 27.9 × 5 = 139.5 damage per turn. Dead in 1 turn.

### Reserve Threshold
FOOD_RESERVE_TURNS = 4. The `give` action only donates surplus above this reserve.
For a pawn: 4 × 2.7 = 10.8 (exceeds capacity 8.7, so pawns give everything above diet).

### Recovery Phases (squad_recover)
1. Each piece harvests and eats independently
2. Overflow pours into bishops first (they're the bank), then others
3. Deficit pieces draw from bishops first, then richest donors

### Pre-March Distribution (pre_march_distribute)
Before each march leg, bishops top up the hungriest squad members.
Donors keep at least 2 turns of food for themselves.
Neediest (by food/capacity ratio) are filled first.

---

## 4. Terrain & Combat Interactions

### Elevation (Primary)
- **Sight:** Higher elevation = longer sight range. `effective_sight = base + elev × 0.005`
- **Movement cost:** Uphill costs more than flat or downhill (`_move_cost` function)
- **Implicit defense:** High ground = see attackers coming + movement cost advantage

### Food Tiers (Primary)
- Computed by FoodYield from terrain + basins (river proximity, soil, etc.)
- 0 = barren, 7 = lush. Most strategic decisions revolve around tier distribution.
- ~55% of land hexes are tier 0–1 (food deserts)

### Climate (Secondary — Future)
- Modifiers on top of existing systems, NOT a new system
- Desert: reduce food tier, increase movement cost
- Jungle: slow movement, high food tier
- Tundra: reduce harvest efficiency
- Note: many "climate" effects already emerge from elevation + food tier combinations.
  High elevation + low food = tundra-like. Low elevation + high food = fertile valley.
  Climate tags may be primarily cosmetic labels for existing terrain combos.

### Facing & Sight Cones
- Each piece has a facing direction (0–5 on hex grid)
- `give` action: can only give to allied pieces in the facing cone
- `pieces_in_sight`: visibility check using facing + elevation
- Knights can exploit blind spots; queens on high ground have no blind spots

---

## 5. Squad Compositions & Tactics

### Composition 1: Queen's Entourage
**Pieces:** 1 Queen + 2–3 Bishops
**Purpose:** The main strike force. Bishops bankroll the queen's massive appetite.
**Tactic:** Bishops carry ~360–540 food. Queen marches with impunity while bishops
redistribute. Vulnerable to knight raids on the bishops.

### Composition 2: Pawn Work Party
**Pieces:** 4–6 Pawns + 1 Bishop (safety net)
**Purpose:** Establish and man waystations along the march route.
**Tactic:** Pawns advance to a fertile hex, set up harvest+give patrol loops.
Bishop provides emergency food buffer if the hex is marginal. Self-sustaining
on tier 2+ terrain.

### Composition 3: Bishop Trucking Convoy
**Pieces:** 2–4 Bishops
**Purpose:** Shuttle food from rear waystations to the front line.
**Tactic:** Bishop fills up at a pawn waystation, marches to the queen's
position, pre_march_distribute dumps food into the queen's entourage,
bishop returns for refill. Round-trip radius ~47 turns per bishop.
Can relay between bishops for longer supply lines.

### Composition 4: Waystation Garrison
**Pieces:** 3–4 Pawns + 1 Rook (once built)
**Purpose:** Permanent defended supply point.
**Tactic:** Pawns harvest continuously. Rook provides defense umbrella.
Bishop truckers dock here to refill. A knight raiding this needs to deal
with the rook's defense — expensive and slow, giving time for reinforcements.

### Composition 5: Knight Raider Pack
**Pieces:** 2–3 Knights
**Purpose:** Disrupt enemy supply lines, scout waystation locations.
**Tactic:** Speed 4 lets them outrun everything except other knights.
Target priority: unguarded bishops (food jackpot), pawn waystations
(cut supply), enemy scouts. Avoid queens on high ground at all costs.

### Composition 6: Full March Column
**Pieces:** 1 Queen + 2 Bishops + 5 Pawns + 1–2 Knights
**Purpose:** The complete campaign force.
**Tactic:**
1. Knights scout ahead, identify next waystation location
2. Pawns advance and establish waystation
3. Queen marches to waystation (bishop-funded)
4. Queen builds rook to fortify waystation
5. Repeat toward enemy capital
6. Bishop truckers maintain supply line to rear

### The Campaign Arc
```
Home Settlement
    │
    ├── Waystation 1 (pawns + rook) ←── bishop trucker loop
    │       │
    │       ├── Waystation 2 (pawns + rook) ←── bishop trucker loop
    │       │       │
    │       │       ├── Waystation 3 (pawns + rook)
    │       │       │       │
    │       │       │       └── Queen's entourage (front line)
    │       │       │               ↑
    │       │       │           knight scouts
    │       │       │
    │       │       └── bishop truckers refill here
    │       │
    │       └── bishop truckers refill here
    │
    └── King stays here (safe, activates eggs)
```

---

## 6. Spawning (Design Sketch — Separate Notebook)

### Egg Mechanic
- **Queen lays eggs** at a certain rate (tunable — per-turn? per-N-turns?)
- **Eggs are inert** until "activated" — queen must be near the king OR a
  home settlement
- **Activation** produces a piece type (pawn by default? or chosen?)
- **Deployment:** Queen can carry activated eggs and deploy them in the field

### Rook Construction
- **Queen builds rooks**, not recruits them
- **Cost:** TBD — food? Turns stationary? A special resource?
- **Rooks are immobile (or speed 1)** once placed
- **Can rooks be destroyed?** If yes, by what threshold of force?
- **Location matters:** A rook on tier 5 is an economic asset (passive harvest).
  A rook on tier 1 is just a roadblock.

### Questions to Resolve
- What determines egg production rate? Queen's food level? Terrain?
- Can eggs be destroyed (e.g. by a knight raid)?
- What piece types can the queen spawn? Just pawns? Or any type at higher cost?
- Is there an upper limit on total pieces per side?
- Does a rook require a pawn sacrifice (upgrade mechanic)?

---

## 7. Bump Mechanic

Higher-ranked pieces can swap places with lower-ranked allies.

### Rules
- Must be same owner (allies only)
- Strictly higher rank (RANK_ORDER)
- Mover pays normal terrain movement cost
- Bumped piece gets recovery instructions prepended:
  1. Rotate to face the hex it was displaced from
  2. FORWARD (blocked this turn if mover is still there, succeeds next turn)
  3. Rotate back to original facing
- Rotations are free (cost 0), so recovery only costs 1 movement point next turn

### Why It Matters
Solves the friendly-blocking problem. The queen can push through her own pawns
instead of getting stuck behind them. The pawn recovers next turn automatically.

---

## 8. Scenario Balance Targets

Tuned via Optuna against these targets (1Q+NB+NP, % of longest route completed):

| Scenario                       | Target | Achieved | Notes                          |
|--------------------------------|--------|----------|--------------------------------|
| Queen alone                    |    5%  |   ~5%    | Nearly dead                    |
| Queen + Bishop                 |   20%  |  ~22%    | Bishop helps but can't harvest |
| Queen + 2 Pawns                |   65%  |  ~63%    | Pawns harvest, no bank         |
| Queen + 3 Pawns (no bishop)    |   75%  | ~100%*   | *Overshoot — needs tuning      |
| Queen + Bishop + 2 Pawns       |   90%  |  ~90%    | Sweet spot                     |
| Queen + Bishop + 4 Pawns       |  100%  | ~100%    | Easy                           |
| 4 Queens + 1 Bishop + 4 Pawns  |   35%  |  ~35%    | 4 queens overwhelm 1 bishop   |
| 4 Queens + 2 Bishops + 8 Pawns |   90%  |  ~90%    | Proper army                    |

### Rock-Paper-Scissors

| Attacker → Target     | Result                                         |
|------------------------|-------------------------------------------------|
| Knight → unguarded waystation | Pawns wiped, supply line cut             |
| Knight → queen on high ground | Knight shredded (vision + attack)        |
| Knight → bishop in open       | Easy prey, huge food prize               |
| Knight → rook + garrison      | Stalled, queen reinforcements arrive     |
| Queen → anything              | Dominates, but starves without support   |
| Pawns (many) → knight         | Zerg rush works if they surround it      |

---

## 9. Path Planning (Current State & Next Steps)

### Current: Leapfrog Planner
1. pre_march_distribute (bishop tops up everyone)
2. march_range (find farthest reachable waypoint with food ≥ 0)
3. Deduct travel food
4. squad_recover at waypoint (harvest + share)
5. pre_march_distribute again
6. Repeat

### Problem
march_range rejects waypoints in food deserts (min_food_tier filter).
Long routes through barren terrain = 0 legs = stuck.

### Next Step: Waystation-Aware Routing
Flip from "plan route, survive it" to "plan route AROUND the food."

1. Build waystation graph — cluster hexes with food tier ≥ N
2. Edges connect waystations within one-leg fuel budget (bishop-boosted)
3. Route through the graph = the "gas station problem"
4. Each hop = one leapfrog leg with guaranteed recovery

This naturally handles both sprint (burn the tank) and sustainable (refuel at
waystations) modes.

---

## 10. Key Files Reference

| File / Function           | Purpose                                          |
|---------------------------|--------------------------------------------------|
| `FoodProfile`             | Tuned diet/capacity/harvest/speed per piece type |
| `FoodYield`               | Compute per-hex food tiers from terrain + basins |
| `FoodSimulator`           | Turn-based sim with interleaved execution        |
| `squad_recover`           | Post-march harvest + surplus sharing             |
| `pre_march_distribute`    | Bishop tops up squad before departure            |
| `feed_cover`              | Pyomo optimizer for feeder assignment             |
| `march_range`             | Farthest reachable waypoint calculation           |
| `leapfrog_plan`           | Chain of march + recover legs                    |
| `supply_line`             | Station feeders along a march path               |
| `balance_objective`       | Optuna objective for piece balancing             |
| `run_scenario`            | Black-box sim for balance testing                |
| `balance.db`              | SQLite Optuna study history                      |

# Pyomo × HexMagic — Integration Notes

## The Chocolate and Peanut Butter

**Chocolate:** Pyomo's declarative optimization — define *what* you want, not *how* to search for it.
**Peanut butter:** HexMagic's composable overlays, `@patch`, `@dataclass`, and FastHTML rendering.

The goal: describe a tactical problem (supply a marching queen, assign formation slots,
plan a retreat) as a **model**, solve it, and get back `{piece_id: InstructionList}` —
the same dict everything else in the system already speaks.

---

## Why Pyomo Over Raw scipy.milp

We already use `scipy.optimize.linear_sum_assignment` (Hungarian) for formation slot
assignment and `scipy.optimize.milp` is imported. These work great for single-matrix
problems. But the supply chain problem has **coupled decisions across time**:

- Who goes where (assignment) affects who can give food to whom (visibility)
- Food at turn t depends on food at turn t-1 (temporal coupling)
- The queen's position changes each turn (moving target)

With scipy.milp you'd hand-build the A matrix, b vector, and variable bounds.
With Pyomo you write:

```python
def food_balance(m, p, t):
    if t == 0: return m.food[p, 0] == initial_food[p]
    return m.food[p, t] == (m.food[p, t-1] - m.diet[p]
        + harvest_if_assigned(m, p, t)
        + received(m, p, t) - given(m, p, t))
m.food_bal = pyo.Constraint(m.P, m.T, rule=food_balance)
```

Same math. Readable. Debuggable. And Pyomo compiles it to the same solver format.

---

## Solver: HiGHS (already installed)

Pyomo needs a solver backend. HiGHS ships with scipy and handles MILP natively.
The bridge is the `appsi_highs` interface in Pyomo, or we can use:

```python
# Option A: appsi_highs (if pyomo detects it)
solver = pyo.SolverFactory('appsi_highs')

# Option B: write to .mps, call scipy.milp manually
# (fallback if appsi doesn't work)

# Option C: install glpk
# !apt-get install -y glpk-utils  # then SolverFactory('glpk')
```

Test which works first — that's the first cell of the new notebook.

---

## The Three Optimization Problems

### Problem 1: Static Supply (where to put feeders)

**Given:** A settlement, N pawns, M candidate fertile hexes, a queen at a fixed location.
**Decide:** Which pawn goes to which hex.
**Objective:** Maximize food delivered to queen per turn.

This is basically the Hungarian assignment we already do, but with richer constraints
(sight cones, terrain, capacity limits). Good **starter problem** for Pyomo.

```
Sets:    F = feeders, C = candidate stations
Vars:    assign[f,c] ∈ {0,1}
Constr:  Σ_c assign[f,c] = 1              (each feeder → one station)
         Σ_f assign[f,c] ≤ station_cap    (don't overcrowd)
Obj:     maximize Σ_f,c assign[f,c] × delivery_rate[f,c]
```

### Problem 2: March Supply Chain (temporal)

**Given:** A march path with queen arrival times, candidate station hexes along it,
feeder pieces with positions/speeds.
**Decide:** Station assignments + food transfer schedule across T turns.
**Objective:** Maximize min(queen food at any turn) — i.e. keep her above starvation.

This is the big one. It's a multi-period LP with binary assignment + continuous food flows.

```
Sets:    P = all pieces, F ⊂ P = feeders, T = turns
Vars:    assign[f,c] ∈ {0,1}, food[p,t] ≥ 0, give[f,p,t] ≥ 0
Constr:  food balance (per piece per turn)
         capacity limits
         sight cone (give only if visible)
         queen position fixed to march_path[t]
Obj:     maximize min_t food[queen, t]
```

Size estimate: 6 pieces × 30 turns = 180 food vars + 6×6×30 = 1080 give vars
+ 6×10 assign vars ≈ 1300 variables. HiGHS eats this for breakfast.

### Problem 3: Mission Replan (mid-sim, fast)

**Given:** Current piece positions, food levels, remaining march path, possibly
a new threat hex (combat).
**Decide:** New assignments, possibly new target.
**Objective:** Same as #2 but from current state, or switch to "maximize
distance from threat" if combat_pressure is high.

This is Problem 2 re-solved with updated initial conditions. The model structure
is identical — only the parameters change. This is where Pyomo shines: you update
`m.food[p, 0] = current_food[p]` and re-solve.

---

## Architecture: How It Fits Together

```
┌──────────────────────────────────────────────────────┐
│  Mission                                              │
│    .target, .formation, .soldiers, .suppliers          │
│    .plan() → calls build_model() + solve()            │
│    .evaluate() → MissionStatus                        │
│    .replan() → rebuild model from current state        │
│    .as_callback() → SquadCallback for FoodSimulator    │
├──────────────────────────────────────────────────────┤
│  SupplyModel (Pyomo wrapper)                          │
│    .build(pieces, stations, path, tiers, turns)       │
│    .solve() → extracts assign[], food[], give[]       │
│    .to_orders() → {piece_id: InstructionList}         │
│    .to_df() → pd.DataFrame of optimal food schedule   │
│    .__ft__() → visual comparison: plan vs actual       │
├──────────────────────────────────────────────────────┤
│  FoodSimulator (existing)                             │
│    executes the plan, fires callbacks each turn        │
│    callbacks can trigger Mission.replan()              │
├──────────────────────────────────────────────────────┤
│  Overlays (existing pattern)                          │
│    MissionOverlay — target, path progress, grade       │
│    SupplyModelOverlay — optimal vs actual food curves  │
│    SquadFoodTrailOverlay — what actually happened      │
└──────────────────────────────────────────────────────┘
```

---

## The FastHTML-Style API Goal

Just like overlays are composable one-liners:

```python
TerrainDisplay(
    CreamOverlay(),
    FoodOverlay(color="#8D6E63"),
    MissionOverlay(mission),
    PieceOverlay([squad]),
    terrain=terr, basins=basins, region=region
)
```

The optimization should feel the same:

```python
# Build + solve in one shot
model = SupplyModel.from_mission(mission)
model.solve()
show(model)  # __ft__ renders optimal food schedule chart

# Extract orders
orders = model.to_orders(grid)
for p in all_pieces:
    if p.id in orders:
        p.instructions = orders[p.id]

# Or even more concise:
mission.plan()  # internally builds model, solves, assigns instructions
sim.run(30)
show(mission)   # __ft__ renders status timeline + grade
```

---

## Key Design Decisions to Make

### 1. Visibility precomputation

The `give[f,p,t]` variable needs to know: can feeder f see piece p at turn t?
For a stationary feeder this is just the sight cone from its station.
For a moving queen, her position changes each turn.

**Approach:** Precompute a binary matrix `can_give[f, c, t]` before building the model.
For each candidate station c, for each turn t, check if `march_path[t]` is in
the sight cone from c. This is O(stations × turns × sight_hexes) — tiny.

### 2. How granular are "turns" in the model?

The sim has sub-turn slots (move_strength budget). The Pyomo model probably
works at **turn granularity** — one harvest + one give per turn per feeder.
This is a simplification but matches the pawn's typical H→G→H→G patrol.

### 3. Fixed queen path vs optimized queen path

Start with **fixed queen path** (from pathfind). The model only optimizes
feeder placement. Later, you could let the model suggest pauses:

```python
# Future: queen can choose to pause at fertile hexes
m.queen_pause = pyo.Var(m.T, domain=pyo.Binary)  # 1 = pause this turn
# ... adjust queen position schedule based on pauses
```

### 4. Model → InstructionList translation

The model outputs `assign[feeder, station]`. Translating to InstructionList:

```python
def to_orders(self, grid):
    orders = {}
    for f in self.feeders:
        for c in self.stations:
            if pyo.value(self.m.assign[f.id, c]) > 0.5:
                # Pathfind feeder → station
                path = f.pathfind(c, grid, self.elevations)
                facing = _facing_toward(grid, c, self.path_hex_for[c])
                rules = path.to_rules(grid, f.facing)
                rules += rotate_to(f.facing, facing)
                rules += [Instruction.HARVEST.value, Instruction.GIVE.value]
                orders[f.id] = InstructionList(rules, patrol=True)
    return orders
```

---

## Notebook Roadmap

### Cell group 1: Solver check
```python
import pyomo.environ as pyo
# Test HiGHS / GLPK / CBC availability
# Tiny 2-variable LP to confirm it works
```

### Cell group 2: Static supply model (Problem 1)
- Use PieceBoard.hilly test board
- Fixed queen, 5 pawns, 10 candidate stations
- Build model, solve, extract assignments
- Compare to Hungarian result — should match or beat

### Cell group 3: Temporal march model (Problem 2)
- Queen marches 20 hexes
- Model decides station placement + food schedule
- `show(model)` — chart of optimal food[queen, t]
- Convert to InstructionLists, run in FoodSimulator
- Compare model prediction vs sim actual (they should match closely)

### Cell group 4: Replan demo (Problem 3)
- Run sim 10 turns
- Inject a "punch" — kill a pawn, or block the path
- Rebuild model from current state
- Show before/after plans

### Cell group 5: Mission integration
- Wire model into Mission.plan()
- Mission.as_callback() triggers replan when status degrades
- Full 40-turn sim with adaptive supply chain
- MissionOverlay on the map

---

## Data Flow Cheat Sheet

```
Terrain + food_tiers + pieces
    │
    ▼
SupplyPlanner.plan_march(path)
    │  (candidate stations, timing, spacing)
    ▼
SupplyModel.build(pieces, stations, path, turns)
    │  (Pyomo model: sets, params, vars, constraints, objective)
    ▼
SupplyModel.solve()
    │  (HiGHS finds optimal assignment + food schedule)
    ▼
SupplyModel.to_orders(grid)
    │  ({piece_id: InstructionList})
    ▼
FoodSimulator.run()
    │  (executes orders, fires callbacks)
    ▼
Mission.evaluate() → MissionStatus
    │
    ├─ grade OK → continue
    └─ needs_replan → rebuild model from current state → new orders
```

---

## What We Keep vs What Changes

| Component | Status | Notes |
|-----------|--------|-------|
| FoodSimulator | ✅ keep | Execution engine, no changes needed |
| SupplyPlanner | 🔄 refactor | `.plan_march()` becomes input to Pyomo model |
| supply_line (Squad patch) | ❌ replace | Greedy heuristic → Pyomo optimal |
| Hungarian matching | ✅ keep for formations | Formation slots are a pure assignment problem |
| sync_paths | ✅ keep | Pyomo handles food timing, sync handles arrival timing |
| MissionStatus | ✅ keep | Evaluation is independent of planning method |
| Overlays | ✅ keep + add | Add SupplyModelOverlay for plan visualization |
| food_triage callback | 🔄 evolve | Becomes the "replan" trigger inside Mission |

---

## One More Thing: The AMPL Aesthetic

The real win isn't just "better solutions" — it's that the **problem statement
becomes the documentation**. When someone reads:

```python
m.food_bal = Constraint(m.P, m.T, rule=lambda m, p, t:
    m.food[p,t] == m.food[p,t-1] - m.diet[p] + m.harvest[p,t] + m.received[p,t])
```

They understand the food system. No need to trace through simulator loops.
The model IS the spec. The simulator validates it. The overlays visualize it.
That's the chocolate and peanut butter. 🍫🥜


# Combat System Design

## Overview

Combat extends the FoodSimulator pattern: a turn-based simulator with callbacks, event logging, and integration with the existing piece stats and movement budget system. The key differentiator is **high ground**—elevation grants significant combat advantage.

---

## Core Mechanics

esults from `HexMagic/game/combat_balance.py` — a turn-based battle simulator for balancing piece stats.

## Design Goals

- **Rook**: High defense, low offense (tank)
- **Knight**: High offense, low defense (glass cannon)
- **Queen**: Balanced powerhouse
- **Target**: 1 Queen ≈ 1 Rook + 2 Knights

## Final Tuned Stats

| Piece  | Offense | Defense | Health | Role |
|--------|---------|---------|--------|------|
| QUEEN  | 8.4     | 1.29    | 121    | Powerhouse |
| ROOK   | 2.5     | 1.6     | 130    | Tank |
| KNIGHT | 7.5     | 0.5     | 55     | Glass cannon |
| BISHOP | 4.0     | 1.0     | 100    | Balanced |
| PAWN   | 1.5     | 0.8     | 80     | Worker |
| KING   | 3.0     | 1.5     | 100    | Protected |

## Battle Results (500 runs, 15% damage variance)

| Matchup | Winner | Winrate | Avg Turns |
|---------|--------|---------|-----------|
| Q vs R+2K | Queen | **60.4%** | 32.7 |
| Q vs R | Queen | 100% | 25.3 |
| Q vs 2K | Queen | 100% | 7.9 |
| R vs 2K | Knights | 100% | 17.7 |
| R vs K | Rook | 100% | 11.5 |
| Q vs 2R | Rooks | 100% | 38.7 |

## Key Findings

1. **Q vs R+2K at ~60%** — Close to the 50% target. The Queen has a slight edge, which feels right for a "powerhouse" unit.

2. **Knights punish Rooks** — 2 Knights beat a Rook 100% of the time. Glass cannons melt the tank before it can deal enough damage back.

3. **Rook beats single Knight** — In 1v1, the Rook's tankiness outlasts the Knight's burst.

4. **2 Rooks beat Queen** — Two tanks working together overwhelm the Queen.

5. **Damage variance matters** — Without variance, outcomes are deterministic. The 15% variance creates interesting probabilistic outcomes.

## Combat Mechanics

### Damage Formula
```
actual_damage = raw_offense / target_defense
```

### Turn Structure
1. All Team 0 units attack (simultaneously)
2. All Team 1 units attack (simultaneously)
3. Check for victory
4. Repeat

### Target Priority
Default: `lowest_hp` (focus fire on weakest enemy)

Options: `random`, `highest_offense`

## Usage

```python
from HexMagic.game.combat_balance import (
    test_default_balance,
    run_scenario_batch,
    scenario_custom,
    run_battle,
    COMBAT_PROFILES
)

# Run all standard scenarios
test_default_balance(runs=100)

# Custom matchup
summary = run_scenario_batch(
    ['QUEEN', 'PAWN'], 
    ['ROOK', 'KNIGHT', 'KNIGHT'],
    runs=100
)
print(f"Team0 wins: {summary.team0_winrate:.1%}")

# Single battle with custom variance
team0, team1 = scenario_custom(['QUEEN'], ['ROOK', 'KNIGHT'])
result = run_battle(team0, team1, damage_variance=0.2)
print(f"Winner: Team {result.winner}, Turns: {result.turns}")
```

## Tuning Notes

The balance is **sensitive** to small stat changes. During tuning:
- Queen offense 8.0 → 0% vs R+2K
- Queen offense 8.5 → 92% vs R+2K
- Queen offense 8.4 → 60% vs R+2K (sweet spot)

This sharp threshold means the "1Q = 1R + 2K" balance requires precise tuning. The `grid_search_balance()` function can help find optimal values if you want different balance points.

### Combat Range

Base combat range depends on piece type, modified by elevation:

```
combat_range = base_range + floor(elevation / elevation_delta)
```

Where `elevation_delta` is the terrain's elevation step (typically ~100m). A piece at elevation 500 on a map with 100m deltas gets +5 hex range. This makes hilltops and ridgelines strategically vital.

**Base ranges by type:**
- PAWN: 1 (melee worker, can defend but shouldn't fight)
- KNIGHT: 2 (fast scout, hit-and-run)
- ROOK: 3 (siege unit, stationary firepower)
- BISHOP: 2 (balanced, mobile support)
- QUEEN: 3 (powerful all-rounder)
- KING: 1 (moderate, stays protected)

### Damage Calculation

Damage dealt per attack:

```python
base_damage = attacker.attack_strength * attacker.size / 100
terrain_mod = 1.0 + (attacker_elevation - defender_elevation) * 0.1
size_ratio = min(1.5, attacker.size / max(1, defender.size))
damage = base_damage * terrain_mod * size_ratio
```

**PIECE_DEFAULTS reminder:** (attack, move, harvest, scout)
- PAWN: (1, 3, 7, 2) — weak attack
- KNIGHT: (2, 7, 1, 5) — scout, moderate attack
- ROOK: (7, 2, 1, 1) — **strongest attack**, slow
- BISHOP: (3, 4, 4, 4) — balanced
- QUEEN: (6, 5, 3, 5) — second strongest
- KING: (4, 2, 2, 3) — moderate defense

### High Ground Advantage

Elevation provides three bonuses:

1. **Extended Range**: +1 hex per elevation_delta (multiplicative with sight)
2. **Damage Bonus**: +10% per elevation_delta above target
3. **Defense Bonus**: Harder to hit from below (attackers at -10% per delta)

A unit on a 3-delta hill attacking down gets +30% damage and 3 extra range hexes.

### Resolution Order

Combat resolves by `RANK_ORDER` (King → Queen → Rook → Bishop → Knight → Pawn), then by food (hungrier pieces act first—desperation):

```python
RANK_ORDER = {
    PieceType.KING:   0,  # acts first (priority target, protected)
    PieceType.QUEEN:  1,
    PieceType.ROOK:   2,
    PieceType.BISHOP: 3,
    PieceType.KNIGHT: 4,
    PieceType.PAWN:   5,  # acts last
}

def combat_order(pieces: list[Piece]) -> list[Piece]:
    return sorted(pieces, key=lambda p: (
        RANK_ORDER.get(p.piece_type, 99), 
        p.food,  # hungrier acts first
        p.id     # tiebreaker
    ))
```

---

## CombatSimulator Design

Extends FoodSimulator's callback-driven architecture:

```python
@dataclass
class CombatEvent:
    turn: int
    slot: int
    event_type: CombatEventType  # ATTACKED, DEFENDED, RETREATED, KILLED, etc.
    attacker_id: str
    defender_id: str
    damage: float
    attacker_hex: int
    defender_hex: int
    elevation_diff: int

CombatCallback = Callable[[CombatEvent], None]

@dataclass
class CombatSimulator:
    """Turn-based combat with interleaved food/combat phases."""
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    pieces: list[Piece]
    countries: np.ndarray = None
    elevation_mult: float = 0.005
    elevation_delta: float = 100.0  # meters per combat range bonus
    
    squads: list[Squad] = field(default_factory=list)
    callbacks: dict = field(default_factory=dict)  # squad.id → CombatCallback
    
    _rows: list = field(default_factory=list)
    turn: int = 0
```

### Event Types

```python
class CombatEventType(Enum):
    # Existing from SimEventType
    MOVED     = "moved"
    BLOCKED   = "blocked"
    ROTATED   = "rotated"
    HARVESTED = "harvested"
    GAVE      = "gave"
    ATE       = "ate"
    STARVED   = "starved"
    DIED      = "died"
    
    # Combat-specific
    ATTACKED   = "attacked"      # dealt damage
    DEFENDED   = "defended"      # took damage but survived
    ROUTED     = "routed"        # forced retreat
    KILLED     = "killed"        # reduced to 0 health
    CAPTURED   = "captured"      # special: King taken
    FLANKED    = "flanked"       # attacked from non-facing direction
```

### Turn Structure

Each turn follows this sequence:

```
1. Movement phase (existing)
   - Resolve by RANK_ORDER
   - Consume move budget
   
2. Combat phase (new)
   - Build target lists (enemies in range)
   - Resolve by RANK_ORDER
   - Apply damage, check deaths
   - Fire callbacks
   
3. Supply phase (existing)
   - Harvest
   - Give
   - Eat
   - Check starvation deaths
   
4. End-of-turn callbacks
```

### Combat Resolution

```python
def _combat_phase(self):
    """Resolve all combat for the current turn."""
    attackers = combat_order([p for p in self.alive if p.health > 0])
    
    for attacker in attackers:
        if attacker.health <= 0:
            continue  # died earlier this phase
            
        targets = self._find_targets(attacker)
        if not targets:
            continue
            
        # Target selection: lowest health enemy in range
        target = min(targets, key=lambda p: (p.health, p.id))
        damage = self._calc_damage(attacker, target)
        
        target.health -= damage
        self._log_combat(attacker, target, damage)
        
        if target.health <= 0:
            target.health = 0
            self._log_combat(attacker, target, 0, CombatEventType.KILLED)

def _find_targets(self, piece: Piece) -> list[Piece]:
    """Find enemy pieces within combat range."""
    piece_elev = self.elevations[piece.location]
    bonus_range = int(piece_elev / self.elevation_delta)
    total_range = BASE_COMBAT_RANGE[piece.piece_type] + bonus_range
    
    in_range = set()
    self._flood_range(piece.location, total_range, in_range)
    
    return [p for p in self.alive 
            if p.owner_id != piece.owner_id 
            and p.location in in_range]

def _calc_damage(self, attacker: Piece, defender: Piece) -> float:
    """Calculate damage with elevation modifiers."""
    atk_elev = self.elevations[attacker.location]
    def_elev = self.elevations[defender.location]
    elev_diff = (atk_elev - def_elev) / self.elevation_delta
    
    base = attacker.attack_strength * attacker.size / 100
    terrain_mod = 1.0 + elev_diff * 0.1
    size_ratio = min(1.5, attacker.size / max(1, defender.size))
    
    return max(1.0, base * terrain_mod * size_ratio)
```

---

## DEFEND Instruction

The existing `Instruction.DEFEND` (value 1) gains combat meaning:

```python
def _is_defending(self, piece: Piece) -> bool:
    """Check if piece is in DEFEND stance this turn."""
    il = piece.instructions
    if not il.rules:
        return False
    return Instruction(il.rules[il.cursor]) == Instruction.DEFEND

DEFEND_BONUS = 0.5  # take 50% less damage when defending
```

When defending:
- Movement budget is consumed (costs 1 slot)
- Piece does not move
- Incoming damage reduced by 50%
- +1 to combat range (fortified position)

---

## Flank Attacks

Attacking from outside a piece's facing direction grants bonus damage:

```python
def _is_flanking(self, attacker: Piece, defender: Piece) -> bool:
    """True if attacker is outside defender's forward arc."""
    # Defender's forward arc: facing ± 1 direction
    forward_arc = {defender.facing, (defender.facing + 1) % 6, (defender.facing - 1) % 6}
    
    # Direction from defender to attacker
    direction = self._hex_direction(defender.location, attacker.location)
    return direction not in forward_arc

FLANK_BONUS = 1.3  # +30% damage from flanks/rear
```

This makes facing and scouting strategically important—KNIGHT's high scout stat helps detect ambushes.

---

## Retreat & Rout

When a piece takes critical damage (below 25% health), it may rout:

```python
ROUT_THRESHOLD = 0.25  # 25% health

def _check_rout(self, piece: Piece, damage: float) -> bool:
    """Check if piece routs after taking damage."""
    if piece.health / piece.max_health > ROUT_THRESHOLD:
        return False
    
    # Rout chance based on remaining health and piece type
    base_chance = 0.5 * (1 - piece.health / piece.max_health)
    
    # Higher rank = less likely to rout
    rank_mod = RANK_ORDER.get(piece.piece_type, 5) * 0.1
    
    return random.random() < base_chance + rank_mod
```

Routed pieces:
- Automatically move away from nearest enemy
- Cannot attack this turn
- Recover composure next turn if not attacked

---

## Pyomo Integration: Tactical Optimization

Just like the supply chain optimization, combat can leverage Pyomo for tactical decisions:

### Problem: Optimal Attack Assignment

Given N attackers and M defenders, minimize total damage to friendlies while maximizing damage to enemies:

```python
def build_combat_model(attackers, defenders, grid, elevations, elevation_delta):
    """
    Sets:
        A = attackers
        D = defenders
        
    Params:
        damage[a,d] = predicted damage if a attacks d
        range_ok[a,d] = 1 if d is in range of a
        
    Vars:
        attack[a,d] ∈ {0,1} = 1 if a attacks d
        
    Constraints:
        ∀a: Σ_d attack[a,d] ≤ 1  # each attacker picks one target
        attack[a,d] ≤ range_ok[a,d]  # can only attack in range
        
    Objective:
        max Σ_{a,d} damage[a,d] * attack[a,d] * priority[d]
        
    Where priority[d] favors:
        - Low-health enemies (finish them off)
        - High-rank enemies (target queens/kings)
        - Enemies threatening our high-rank pieces
    """
    m = pyo.ConcreteModel()
    m.A = pyo.Set(initialize=[a.id for a in attackers])
    m.D = pyo.Set(initialize=[d.id for d in defenders])
    
    # Precompute damage and range
    damage_matrix = {}
    range_matrix = {}
    for a in attackers:
        for d in defenders:
            range_matrix[a.id, d.id] = _in_range(a, d, grid, elevations, elevation_delta)
            damage_matrix[a.id, d.id] = _calc_damage(a, d, elevations, elevation_delta)
    
    m.damage = pyo.Param(m.A, m.D, initialize=damage_matrix)
    m.range_ok = pyo.Param(m.A, m.D, initialize=range_matrix)
    
    m.attack = pyo.Var(m.A, m.D, domain=pyo.Binary)
    
    # One target per attacker
    def one_target(m, a):
        return sum(m.attack[a, d] for d in m.D) <= 1
    m.one_target = pyo.Constraint(m.A, rule=one_target)
    
    # Range constraint
    def in_range(m, a, d):
        return m.attack[a, d] <= m.range_ok[a, d]
    m.in_range = pyo.Constraint(m.A, m.D, rule=in_range)
    
    # Objective: maximize expected damage dealt
    def objective(m):
        return sum(m.damage[a, d] * m.attack[a, d] 
                   for a in m.A for d in m.D)
    m.obj = pyo.Objective(rule=objective, sense=pyo.maximize)
    
    return m
```

### Problem: Defensive Positioning

Where should pieces move to maximize defensive coverage of high-value targets (links to `warehouselocation.md`):

```python
# Inputs:
#   - High-value hexes (settlements, resources, chokepoints)
#   - Current enemy positions
#   - Terrain elevations

# Objective:
#   max(coverage of high-value hexes) + elevation_bonus - exposure_to_enemy
```

---

## Callbacks & Replanning

Following the FoodSimulator pattern, combat supports callbacks for adaptive AI:

```python
@dataclass
class CombatReport:
    """End-of-combat-phase context for callbacks."""
    turn: int
    squad: Squad
    enemies_killed: list[str]
    friendlies_killed: list[str]
    damage_dealt: float
    damage_taken: float
    routed: list[str]
    
def combat_callback_example(report: CombatReport):
    """Example: retreat if losing badly."""
    if report.damage_taken > report.damage_dealt * 2:
        # Trigger retreat orders
        for piece in report.squad.alive:
            piece.instructions = InstructionList(
                [Instruction.ROT_L.value] * 3 + [Instruction.FORWARD.value] * 5,
                patrol=False
            )
```

The Mission system can use combat callbacks to trigger replanning:

```python
class Mission:
    def combat_callback(self) -> CombatCallback:
        def cb(report: CombatReport):
            if report.friendlies_killed:
                self.replan()  # rebuild supply/combat orders
        return cb
```

---

## Integration with Existing Systems

### FoodSimulator Extension

CombatSimulator inherits from FoodSimulator, adding the combat phase:

```python
class CombatSimulator(FoodSimulator):
    def run(self, turns: int):
        for _ in range(turns):
            self._movement_phase()
            self._combat_phase()      # NEW
            self._harvest_phase()
            self._give_phase()
            self._eat_phase()
            self._end_of_turn_callbacks()
            self.turn += 1
```

### PieceBoard Testing

Extend PieceBoard for combat testing scenarios:

```python
class CombatBoard(PieceBoard):
    """Test board with enemy pieces for combat scenarios."""
    
    def add_enemy_squad(self, count: int, positions: list[int]):
        """Add enemy pieces at specified positions."""
        ...
    
    def run_combat_sim(self, turns: int) -> pd.DataFrame:
        """Run combat simulation, return event log."""
        ...
```

In [ ]:
#| default_exp game/mechanics/simulator

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import FlowOverlay, CreamOverlay, RiverOverlay, ClimateOverlay, OverlayContext
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns
from HexMagic.plot.cube import HexPosition


In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc

import httpx
import random
import pandas as pd
import threading
from dataclasses import dataclass
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import math
from dataclasses import dataclass, field
from functools import cached_property
#| export
import heapq
import io
from monsterui.all import *
from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds

In [ ]:
#| export

from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging

from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad
from HexMagic.game.data import RANK_ORDER 

from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan
from HexMagic.game.piece import SquadPlanOverlay, PieceList, FormationType, SurroundMode, TroopPath, _facing_toward
from HexMagic.game.piece import GroupedPieceList, SquadPlanOverlay, piece_plan_overlay, SquadSymbolOverlay #, SquadVisionOverlay
from HexMagic.game.piece import FormationType, _formation_offsets


from HexMagic.game.piece import pieces_center, StatBar, SquadVisionOverlay, GameParts,SettlementOverlay, KingdomNamesOverlay, PieceOverlay

from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds




from enum import Enum

In [ ]:
#| export
from HexMagic.game.food import FoodOverlay, FoodYield, PieceOverlay, PieceDataOverlay, PieceArrowOverlay, SquadSymbolOverlay

In [ ]:
import pyomo.environ as pyo
import optuna

Based on your integration notes, you need:

```bash
!pip install pyomo highspy
```

- **`pyomo`** — the modeling library itself
- **`highspy`** — the Python bindings for HiGHS, which lets Pyomo use it via `SolverFactory('appsi_highs')`

You already have `scipy` (which bundles HiGHS internally), but Pyomo talks to HiGHS through `highspy` directly rather than through scipy's wrapper.

After installing, verify with a quick smoke test:

```python
import pyomo.environ as pyo

m = pyo.ConcreteModel()
m.x = pyo.Var(bounds=(0, 10))
m.obj = pyo.Objective(expr=m.x, sense=pyo.maximize)

solver = pyo.SolverFactory('appsi_highs')
result = solver.solve(m)
print(f"Status: {result.solver.termination_condition}, x = {pyo.value(m.x)}")
```

If that prints `Status: optimal, x = 10.0` you're good to go.

In [ ]:
#| export
STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

In [ ]:
showDemo = False
myStuff = GameParts()

In [ ]:

TerrainDisplay(
    CreamOverlay(stylized=True),
    SquadSymbolOverlay([myStuff.queenGuard]),
    #SettlementOverlay(scale=1),
    #KingdomNamesOverlay(),
    PieceOverlay([myStuff.queenGuard]),
    SquadVisionOverlay([myStuff.queenGuard]),
    terrain=myStuff.terr,
    region=myStuff.region(), padding=2,
    board=myStuff.board,
    debug = not showDemo
)



In [ ]:
# Make sure soil exists

soil = SoilSystem.from_plates(myStuff.terr, [])

fy = FoodYield(myStuff.terr, myStuff.basin)
fy.compute()
#show(fy)
myStuff.fy = fy
fy.summary()

In [ ]:
show(fy)

## FoodOverlay

In [ ]:

myStuff.clearTerr()
myStuff.terr.repalette()
TerrainDisplay(
    TerrainOverlay(),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    RiverOverlay(max_width=6),
    terrain=myStuff.terr,
    basins=myStuff.basin,
    debug = not showDemo
)


In [ ]:
#!cat ../../HexMagic/game/piece.py

In [ ]:
#| export
STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

## TroopPath

In [ ]:
myStuff.clearTerr()
myStuff.terr.repalette()
myStuff.grid.adjustRadius(35)


In [ ]:
showDemo = False

In [ ]:
??TerrainOverlay

In [ ]:
#read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

## The simulation

In [ ]:
#| export
from typing import Callable

@dataclass
class TurnReport:
    """End-of-turn context passed to squad callbacks."""
    turn: int
    squad: Squad
    snapshot: dict
    food_audit: dict
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    countries: np.ndarray | None
    elevation_mult: float
    sim_df: pd.DataFrame

SquadCallback = Callable[[TurnReport], None]


In [ ]:
#| export
# Add to SimEventType
class SimEventType(Enum):
    MOVED     = "moved"
    BLOCKED   = "blocked"
    BUMPED    = "bumped"      # ← new
    ROTATED   = "rotated"
    HARVESTED = "harvested"
    GAVE      = "gave"
    ATE       = "ate"
    STARVED   = "starved"
    DIED      = "died"
    ATTACKED  = "attacked"
    COUNTERED = "countered"


_FREE = {Instruction.ROT_L, Instruction.ROT_R}
_COSTS_ONE = {Instruction.PAUSE, Instruction.DEFEND,
              Instruction.HARVEST, Instruction.GIVE, Instruction.SETTLE}

@patch
def _eat_phase(self: FoodSimulator, acted_ids: set = None):
    for piece in list(self.pieces):
        if piece.health <= 0:
            continue

        shortfall = max(0.0, piece.diet - piece.food)
        piece.food = max(0.0, piece.food - piece.diet)
        self._log(piece, SimEventType.ATE, amount=piece.diet)

        if shortfall > 0:
            damage = shortfall * STARVE_MULT
            piece.health -= damage
            self._log(piece, SimEventType.STARVED, amount=damage)

        if piece.health <= 0:
            piece.health = 0
            self._log(piece, SimEventType.DIED)


In [ ]:
#| export
@dataclass
class FoodSimulator:
    """Turn-based simulator with per-slot interleaved execution."""
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    pieces: list
    countries: np.ndarray = None
    elevation_mult: float = 0.005

    squads: list = field(default_factory=list)
    callbacks: dict = field(default_factory=dict)  # squad.id → SquadCallback

    _rows: list = field(default_factory=list)
    turn: int = 0
    snapshots: list = field(default_factory=list)

    def __post_init__(self):
        self._squad_map = {s.id: s for s in self.squads}

    @property
    def alive(self):
        return [p for p in self.pieces if p.health > 0]

    @property
    def occupied(self):
        return {p.location for p in self.alive
                if p.location is not None and p.location >= 0}

    @property
    def df(self) -> pd.DataFrame:
        return pd.DataFrame(self._rows)

    def _log(self, piece, etype: SimEventType, *,
             slot=None, src_hex=None, dst_hex=None,
             dst_piece_id=None, amount=0.0):
        loc = piece.location if piece.location is not None else -1
        self._rows.append({
            'turn': self.turn, 'slot': slot, 'event': etype.value,
            'piece_id': piece.id, 'piece_name': piece.name,
            'src_hex': src_hex if src_hex is not None else loc,
            'dst_hex': dst_hex if dst_hex is not None else loc,
            'dst_piece_id': dst_piece_id, 'amount': amount,
        })

    def _sort_move(self, pieces):
        return sorted(pieces, key=lambda p: (
            RANK_ORDER.get(p.piece_type, 99), p.birth_year, -p.health, p.id))

    # ── actions (unchanged) ──
    def _do_harvest(self, piece, slot=None):
        loc = piece.location
        if loc is None or loc < 0: return
        tier = int(self.food_tiers[loc])
        gained = piece.harvest_strength * tier
        old = piece.food
        piece.food = min(piece.food + gained, piece.food_capacity)
        self._log(piece, SimEventType.HARVESTED, slot=slot, amount=piece.food - old)

    def _do_give(self, piece, slot=None):
        reserve = FOOD_RESERVE_TURNS * piece.diet
        surplus = max(0.0, piece.food - reserve)
        if surplus <= 0: return
        visible = set(piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=self.elevation_mult, facing_only=True))
        receivers = sorted(
            [p for p in self.alive if p is not piece
             and p.owner_id == piece.owner_id
             and p.food < p.food_capacity
             and p.location in visible],
            key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))
        for recv in receivers:
            if surplus <= 0: break
            amount = min(surplus, recv.food_capacity - recv.food)
            recv.food += amount; piece.food -= amount; surplus -= amount
            self._log(piece, SimEventType.GAVE, slot=slot,
                      dst_piece_id=recv.id, amount=amount)


    def _eat_phase(self, acted_ids: set = None):
        for piece in list(self.pieces):
            if piece.health <= 0: continue
            shortfall = max(0.0, piece.diet - piece.food)
            piece.food = max(0.0, piece.food - piece.diet)
            self._log(piece, SimEventType.ATE, amount=piece.diet)
            if shortfall > 0:
                damage = shortfall * STARVE_MULT
                piece.health -= damage
                self._log(piece, SimEventType.STARVED, amount=damage)
            if piece.health <= 0:
                piece.health = 0
                self._log(piece, SimEventType.DIED)

    def _snapshot(self):
        alive = self.alive
        self.snapshots.append({
            'turn': self.turn,
            'alive': len(alive), 'total': len(self.pieces),
            'avg_health': np.mean([p.health for p in alive]) if alive else 0,
            'min_health': min((p.health for p in alive), default=0),
            'avg_food': np.mean([p.food for p in alive]) if alive else 0,
            'avg_turns_left': np.mean([p.food / max(p.diet, 0.01) for p in alive]) if alive else 0,
            'piece_states': {
                p.id: {'location': p.location, 'food': p.food,
                       'food_capacity': p.food_capacity, 'health': p.health}
                for p in self.pieces},
        })

    def _fire_callbacks(self):
        if not self.callbacks: return
        snap = self.snapshots[-1] if self.snapshots else {}
        for squad_id, cb in self.callbacks.items():
            squad = self._squad_map.get(squad_id)
            if squad is None or not squad.alive: continue
            report = TurnReport(
                turn=self.turn, squad=squad, snapshot=snap,
                food_audit=squad.food_audit(self.food_tiers),
                grid=self.grid, elevations=self.elevations,
                food_tiers=self.food_tiers, countries=self.countries,
                elevation_mult=self.elevation_mult, sim_df=self.df,
            )
            cb(report)


    def run(self, num_turns=20):
        for _ in range(num_turns):
            if not self.alive: self._snapshot(); break
            self.step()
        return self

    def summary(self):
        a = self.alive
        df = self.df
        print(f"Turn {self.turn}: {len(a)}/{len(self.pieces)} alive")
        if a:
            print(f"  Health: avg={np.mean([p.health for p in a]):.0f} "
                f"min={min(p.health for p in a):.0f}")
            print(f"  Food:   avg={np.mean([p.food for p in a]):.1f} "
                f"turns={np.mean([p.food/max(p.diet,.01) for p in a]):.1f}")

        if df.empty: return

        # Moves per piece
        moves = df[df['event'] == 'moved'].groupby('piece_name').size()

        # Deaths
        deaths = df[df['event'] == 'died'][['turn', 'piece_name']]

        if not moves.empty:
            print("  Movement:")
            for name, n in moves.items():
                print(f"    {name}: {n} hexes")

        if not deaths.empty:
            print("  Deaths:")
            for _, row in deaths.iterrows():
                print(f"    {row.piece_name} died on turn {row.turn}")


could summary include when pieces died and how far they traveled?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! You can pull both from the event log. Here's an enhanced `summary`:

```python
def summary(self):
    a = self.alive
    df = self.df
    print(f"Turn {self.turn}: {len(a)}/{len(self.pieces)} alive")
    if a:
        print(f"  Health: avg={np.mean([p.health for p in a]):.0f} "
              f"min={min(p.health for p in a):.0f}")
        print(f"  Food:   avg={np.mean([p.food for p in a]):.1f} "
              f"turns={np.mean([p.food/max(p.diet,.01) for p in a]):.1f}")

    if df.empty: return

    # Moves per piece
    moves = df[df['event'] == 'moved'].groupby('piece_name').size()

    # Deaths
    deaths = df[df['event'] == 'died'][['turn', 'piece_name']]

    if not moves.empty:
        print("  Movement:")
        for name, n in moves.items():
            print(f"    {name}: {n} hexes")

    if not deaths.empty:
        print("  Deaths:")
        for _, row in deaths.iterrows():
            print(f"    {row.piece_name} died on turn {row.turn}")
```

The key ideas:

- **Distance**: count rows where `event == 'moved'` grouped by piece — each MOVED event is one hex traversed
- **Deaths**: filter for `event == 'died'` and grab the turn number

You could also add start/end hex distance (crow-flies) if you want strategic distance vs total movement. That'd need `self.grid.distance(start_hex, end_hex)` using the first and last `src_hex` for each piece in the log.

In [ ]:
#| export
@patch
def _try_bump(self: FoodSimulator, mover, blocker, occupied, slot=None):
    """Swap mover (higher rank) with blocker (lower rank).
    
    Prepends recovery instructions to blocker:
      rotate → face old hex, FORWARD, rotate → restore original facing.
    Returns movement cost for the mover, or None if bump isn't allowed.
    """
    # Must be allies
    if mover.owner_id != blocker.owner_id:
        return None
    # Strictly higher rank
    if RANK_ORDER.get(mover.piece_type, 99) <= RANK_ORDER.get(blocker.piece_type, 99):
        return None
    
    old_mover_loc = mover.location
    old_blocker_loc = blocker.location
    old_blocker_facing = blocker.facing
    
    # ── Swap positions ──
    occupied.discard(old_mover_loc)
    occupied.discard(old_blocker_loc)
    mover.location = old_blocker_loc
    blocker.location = old_mover_loc
    occupied.add(old_blocker_loc)
    occupied.add(old_mover_loc)
    
    cost = _move_cost(self.elevations, old_mover_loc, old_blocker_loc)
    
    # ── Build recovery instructions for the bumped piece ──
    # Step 1: rotate to face old position (where it wants to return)
    target_facing = _facing_toward(self.grid, old_mover_loc, old_blocker_loc)
    
    diff = (target_facing - old_blocker_facing) % 6
    if diff <= 3:
        rot_to = [Instruction.ROT_R.value] * diff
    else:
        rot_to = [Instruction.ROT_L.value] * (6 - diff)
    
    # Step 2: FORWARD (will likely be blocked this turn, succeeds next)
    
    # Step 3: rotate back to original facing
    diff_back = (old_blocker_facing - target_facing) % 6
    if diff_back <= 3:
        rot_back = [Instruction.ROT_R.value] * diff_back
    else:
        rot_back = [Instruction.ROT_L.value] * (6 - diff_back)
    
    recovery = rot_to + [Instruction.FORWARD.value] + rot_back
    
    # Insert at current cursor (so recovery executes before existing plan)
    il = blocker.instructions
    il.rules = il.rules[:il.cursor] + recovery + il.rules[il.cursor:]
    # cursor stays — recovery is now what's next
    
    # ── Log ──
    self._log(mover, SimEventType.BUMPED, slot=slot,
              src_hex=old_mover_loc, dst_hex=old_blocker_loc,
              dst_piece_id=blocker.id)
    
    return cost


In [ ]:
#| export
@patch
def _try_forward(self: FoodSimulator, piece, occupied, slot=None, budget=float('inf')):
    d = HexPosition.directions()[piece.facing % 6]
    nbr = self.grid.hexposition_to_index(d, piece.location)
    
    passable = (0 <= nbr < len(self.elevations)
                and nbr not in self.grid.invalidRegion
                and self.elevations[nbr] >= 1)
    
    if not passable:
        self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
        return 0.0
    
    if nbr in occupied and nbr != piece.location:
        blocker = next((p for p in self.alive if p.location == nbr), None)
        if blocker:
            bump_cost = self._try_bump(piece, blocker, occupied, slot)
            if bump_cost is not None:
                return bump_cost
        self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
        return 0.0
    
    cost = _move_cost(self.elevations, piece.location, nbr)
    if cost > budget:
        return 0.0  # Can't afford — don't move!
    
    src = piece.location
    occupied.discard(src); piece.location = nbr; occupied.add(nbr)
    self._log(piece, SimEventType.MOVED, slot=slot, src_hex=src, dst_hex=nbr, amount=cost)
    return cost

In [ ]:
#| export
@patch
def _tick_one(self: FoodSimulator, piece, budget, occupied, slot=None):
    il = piece.instructions
    if not il.rules: return None, 0.0
    while True:
        if il.cursor >= len(il.rules):
            if il.patrol: il.cursor = 0
            else: return None, 0.0
        instr = Instruction(il.rules[il.cursor])
        if instr in _FREE:
            piece.facing = ((piece.facing - 1) if instr == Instruction.ROT_L
                            else (piece.facing + 1)) % 6
            self._log(piece, SimEventType.ROTATED, slot=slot)
            il.cursor += 1; continue
        if instr in _COSTS_ONE:
            if budget < 1.0: return None, 0.0
            il.cursor += 1
            if instr == Instruction.HARVEST: self._do_harvest(piece, slot)
            elif instr == Instruction.GIVE: self._do_give(piece, slot)
            return instr, 1.0
        # FORWARD — pass budget so _try_forward won't move if we can't afford it
        if budget < 1.0: return None, 0.0
        cost = self._try_forward(piece, occupied, slot, budget=budget)
        if cost > 0:
            il.cursor += 1
            return Instruction.FORWARD, cost
        return None, 0.0

In [ ]:
#| export
@patch
def _combat_phase(self: FoodSimulator, moved_ids: set, defended_ids: set):
    """Stationary pieces roll temperament; attackers flame all enemies in sight cone."""
    for piece in list(self.alive):
        if piece.health <= 0:
            continue
        if piece.id in moved_ids:
            continue

        if random.randint(0, 99) < getattr(piece, 'temperment', 100):
            continue

        visible = set(piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=self.elevation_mult, facing_only=True))

        enemies = [p for p in self.alive
                   if p.owner_id != piece.owner_id
                   and p.health > 0
                   and p.location in visible]

        if not enemies:
            continue

        for target in enemies:
            damage = piece.attack_strength / max(target.defense, 0.01)
            target.health = max(0, target.health - damage)
            self._log(piece, SimEventType.ATTACKED,
                      dst_piece_id=target.id, amount=damage)

            # Counter-attack: only if defending AND can see attacker
            if target.id in defended_ids and target.health > 0:
                target_vis = set(target.hexes_in_sight(
                    self.grid, self.elevations,
                    elevation_mult=self.elevation_mult, facing_only=True))
                if piece.location in target_vis:
                    counter = target.attack_strength / max(piece.defense, 0.01)
                    piece.health = max(0, piece.health - counter)
                    self._log(target, SimEventType.COUNTERED,
                              dst_piece_id=piece.id, amount=counter)

            if target.health <= 0:
                self._log(target, SimEventType.DIED)
            if piece.health <= 0:
                self._log(piece, SimEventType.DIED)
                break


In [ ]:
#| export


# ── Updated step() — tracks moved/defended, calls combat after eat ──
@patch
def step(self: FoodSimulator):
    alive = self.alive
    if not alive: return
    max_slots = max(p.move_strength for p in alive)
    budgets = {p.id: float(p.move_strength) for p in alive}
    acted = set()
    moved_ids = set()
    defended_ids = set()
    occupied = self.occupied
    order = self._sort_move(alive)

    for slot in range(max_slots):
        for piece in order:
            if piece.health <= 0: continue
            if budgets[piece.id] <= 0: continue
            instr, cost = self._tick_one(piece, budgets[piece.id], occupied, slot=slot)
            if instr is not None:
                budgets[piece.id] -= cost
                acted.add(piece.id)
                if instr == Instruction.FORWARD:
                    moved_ids.add(piece.id)
                elif instr == Instruction.DEFEND:
                    defended_ids.add(piece.id)

    self._eat_phase()
    self._combat_phase(moved_ids, defended_ids)
    self._snapshot()
    self._fire_callbacks()
    self.turn += 1


### Callbacks

In [ ]:
#| export
def food_triage(report: TurnReport):
    """Prepend harvest+give when squad is hungry."""
    avg_left = report.snapshot.get('avg_turns_left', 999)
    if avg_left >= FOOD_RESERVE_TURNS:
        return
    
    alive = [p for p in report.squad.pieces if p.health > 0]
    for piece in alive:
        loc = piece.location
        if loc is None or loc < 0:
            continue
        
        prepend = []
        tier = int(report.food_tiers[loc]) if 0 <= loc < len(report.food_tiers) else 0
        if tier >= 1 and piece.food < piece.food_capacity:
            prepend.append(Instruction.HARVEST.value)
        
        reserve = FOOD_RESERVE_TURNS * piece.diet
        if piece.food > reserve:
            visible = piece.hexes_in_sight(
                report.grid, report.elevations,
                elevation_mult=report.elevation_mult,
                facing_only=True)
            hungry = any(
                p for p in alive
                if p is not piece and p.location in visible
                and p.food < p.food_capacity
            )
            if hungry:
                prepend.append(Instruction.GIVE.value)
        
        if prepend:
            piece.instructions.rules = prepend + piece.instructions.rules


In [ ]:
def compose_callbacks(*fns: SquadCallback) -> SquadCallback:
    """Chain multiple callbacks — each sees mutations from the previous."""
    def combined(report: TurnReport):
        for fn in fns:
            fn(report)
    return combined


### Analysis

#| export
@patch
def __ft__(self: FoodSimulator):
    """Line charts: alive count, health, food turns remaining."""
    if not self.snapshots:
        return P("No turns simulated yet", cls="text-sm opacity-50")

    df = pd.DataFrame(self.snapshots)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3))

    # Alive
    axes[0].plot(df['turn'], df['alive'], 'o-', color='#2ecc71', lw=2, ms=4)
    axes[0].axhline(df['total'].iloc[0], color='#bbb', ls='--', lw=1)
    axes[0].set_title('Alive', fontweight='bold', fontsize=10)
    axes[0].set_xlabel('Turn'); axes[0].set_ylim(bottom=0)
    axes[0].fill_between(df['turn'], df['alive'], alpha=0.15, color='#2ecc71')

    # Health
    axes[1].plot(df['turn'], df['avg_health'], 'o-', color='#e74c3c', lw=2, ms=4, label='avg')
    axes[1].plot(df['turn'], df['min_health'], 's--', color='#c0392b', lw=1, ms=3, alpha=0.6, label='min')
    axes[1].set_title('Health', fontweight='bold', fontsize=10)
    axes[1].set_xlabel('Turn'); axes[1].legend(fontsize=8); axes[1].set_ylim(bottom=0)

    # Food turns remaining
    axes[2].plot(df['turn'], df['avg_turns_left'], 'o-', color='#f39c12', lw=2, ms=4)
    axes[2].axhline(FOOD_RESERVE_TURNS, color='#e67e22', ls=':', lw=1, label=f'reserve ({FOOD_RESERVE_TURNS})')
    axes[2].set_title('Food (turns left)', fontweight='bold', fontsize=10)
    axes[2].set_xlabel('Turn'); axes[2].legend(fontsize=8); axes[2].set_ylim(bottom=0)

    for ax in axes:
        ax.grid(axis='y', alpha=0.3, lw=0.5)
        ax.tick_params(labelsize=8)

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Simulated {self.turn} turns — {len(self.alive)}/{len(self.pieces)} alive",
          cls="text-xs opacity-60 text-center"),
        cls="space-y-1")

In [ ]:
#| export
@patch
def __ft__(self: FoodSimulator):
    """Line charts by piece type + activity breakdown over time."""
    if not self.snapshots:
        return P("No turns simulated yet", cls="text-sm opacity-50")

    pinfo = {p.id: (p.piece_type, p.diet) for p in self.pieces}
    all_ranks = sorted({pt for pt, _ in pinfo.values()},
                       key=lambda p: RANK_ORDER.get(p, 99))

    rank_colors = {
        PieceType.KING:   '#9b59b6', PieceType.QUEEN:  '#e74c3c',
        PieceType.ROOK:   '#3498db', PieceType.BISHOP: '#2ecc71',
        PieceType.KNIGHT: '#f39c12', PieceType.PAWN:   '#95a5a6',
    }

    turns = [s['turn'] for s in self.snapshots]
    rank_data = {pt: {'alive': [], 'avg_health': [], 'avg_turns_left': []}
                 for pt in all_ranks}

    for snap in self.snapshots:
        by_rank = {pt: {'alive': 0, 'healths': [], 'turns_left': []}
                   for pt in all_ranks}
        for pid, state in snap['piece_states'].items():
            pt, diet = pinfo.get(pid, (None, 1.0))
            if pt is None: continue
            if state['health'] > 0:
                by_rank[pt]['alive'] += 1
                by_rank[pt]['healths'].append(state['health'])
                by_rank[pt]['turns_left'].append(state['food'] / max(diet, 0.01))
        for pt in all_ranks:
            d = by_rank[pt]
            rank_data[pt]['alive'].append(d['alive'])
            rank_data[pt]['avg_health'].append(
                np.mean(d['healths']) if d['healths'] else 0)
            rank_data[pt]['avg_turns_left'].append(
                np.mean(d['turns_left']) if d['turns_left'] else 0)

    # ── Activity data — tactical events only (skip routine move/rotate/eat) ──
    tactical_events = [
        SimEventType.HARVESTED, SimEventType.GAVE,
        SimEventType.ATTACKED,  SimEventType.COUNTERED,
        SimEventType.BUMPED,    SimEventType.STARVED, SimEventType.DIED,
    ]
    event_colors = {
        SimEventType.HARVESTED: '#2ecc71', SimEventType.GAVE:      '#1abc9c',
        SimEventType.ATTACKED:  '#e74c3c', SimEventType.COUNTERED: '#c0392b',
        SimEventType.BUMPED:    '#9b59b6', SimEventType.STARVED:   '#f39c12',
        SimEventType.DIED:      '#2c3e50',
    }
    max_turn = max(turns) if turns else 0
    event_counts = {et: np.zeros(max_turn + 1) for et in tactical_events}
    ev_lookup = {et.value: et for et in tactical_events}
    for row in self._rows:
        et = ev_lookup.get(row['event'])
        if et and row['turn'] <= max_turn:
            event_counts[et][row['turn']] += 1

    active = [et for et in tactical_events if event_counts[et].any()]

    # ── Layout: 3 rank charts on top, stacked bar below ──
    fig = plt.figure(figsize=(11, 6.5))
    gs = fig.add_gridspec(2, 3, height_ratios=[1, 0.85], hspace=0.4)
    axes_top = [fig.add_subplot(gs[0, i]) for i in range(3)]
    ax_act = fig.add_subplot(gs[1, :])

    # ── Top row: rank lines ──
    for pt in all_ranks:
        c = rank_colors.get(pt, '#333'); lbl = pt.name.title()
        d = rank_data[pt]
        axes_top[0].plot(turns, d['alive'], 'o-', color=c, lw=2, ms=3, label=lbl)
        axes_top[1].plot(turns, d['avg_health'], 'o-', color=c, lw=2, ms=3, label=lbl)
        axes_top[2].plot(turns, d['avg_turns_left'], 'o-', color=c, lw=2, ms=3, label=lbl)

    axes_top[0].set_title('Alive by Rank', fontweight='bold', fontsize=10)
    axes_top[0].set_xlabel('Turn'); axes_top[0].set_ylim(bottom=0)
    axes_top[1].set_title('Avg Health by Rank', fontweight='bold', fontsize=10)
    axes_top[1].set_xlabel('Turn'); axes_top[1].set_ylim(bottom=0)
    axes_top[2].axhline(FOOD_RESERVE_TURNS, color='#e67e22', ls=':', lw=1,
                        label=f'reserve ({FOOD_RESERVE_TURNS})')
    axes_top[2].set_title('Food (turns left) by Rank', fontweight='bold', fontsize=10)
    axes_top[2].set_xlabel('Turn'); axes_top[2].set_ylim(bottom=0)
    for ax in axes_top:
        ax.legend(fontsize=7, loc='best')
        ax.grid(axis='y', alpha=0.3, lw=0.5); ax.tick_params(labelsize=8)

    # ── Bottom row: stacked bar of tactical events ──
    turn_range = np.arange(max_turn + 1)
    if active:
        bottoms = np.zeros(max_turn + 1)
        for et in active:
            vals = event_counts[et]
            ax_act.bar(turn_range, vals, bottom=bottoms, width=0.8,
                       color=event_colors.get(et, '#999'),
                       label=et.value.title(), alpha=0.85)
            bottoms += vals

    ax_act.set_title('Tactical Actions per Turn', fontweight='bold', fontsize=10)
    ax_act.set_xlabel('Turn'); ax_act.set_ylabel('Count')
    ax_act.legend(fontsize=7, loc='upper right',
                  ncol=min(len(active), 4) if active else 1)
    ax_act.grid(axis='y', alpha=0.3, lw=0.5); ax_act.tick_params(labelsize=8)
    ax_act.set_ylim(bottom=0)

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Simulated {self.turn} turns — {len(self.alive)}/{len(self.pieces)} alive",
          cls="text-xs opacity-60 text-center"),
        cls="space-y-1")


can you rewrite so that FoodSimulator.__ft__ so that it does it by rank (so there would be a line for all of the pawns, a line for all of the queens, etc)?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def __ft__(self: FoodSimulator):
    """Line charts by piece type: alive count, avg health, avg food turns left."""
    if not self.snapshots:
        return P("No turns simulated yet", cls="text-sm opacity-50")

    # Lookup: piece_id → (piece_type, diet)
    pinfo = {p.id: (p.piece_type, p.diet) for p in self.pieces}
    all_ranks = sorted({pt for pt, _ in pinfo.values()},
                       key=lambda p: RANK_ORDER.get(p, 99))

    rank_colors = {
        PieceType.KING:   '#9b59b6',
        PieceType.QUEEN:  '#e74c3c',
        PieceType.ROOK:   '#3498db',
        PieceType.BISHOP: '#2ecc71',
        PieceType.KNIGHT: '#f39c12',
        PieceType.PAWN:   '#95a5a6',
    }

    turns = [s['turn'] for s in self.snapshots]
    # rank → {metric: [val_per_turn]}
    rank_data = {pt: {'alive': [], 'avg_health': [], 'avg_turns_left': []}
                 for pt in all_ranks}

    for snap in self.snapshots:
        by_rank = {pt: {'alive': 0, 'healths': [], 'turns_left': []}
                   for pt in all_ranks}
        for pid, state in snap['piece_states'].items():
            pt, diet = pinfo.get(pid, (None, 1.0))
            if pt is None: continue
            if state['health'] > 0:
                by_rank[pt]['alive'] += 1
                by_rank[pt]['healths'].append(state['health'])
                by_rank[pt]['turns_left'].append(state['food'] / max(diet, 0.01))

        for pt in all_ranks:
            d = by_rank[pt]
            rank_data[pt]['alive'].append(d['alive'])
            rank_data[pt]['avg_health'].append(
                np.mean(d['healths']) if d['healths'] else 0)
            rank_data[pt]['avg_turns_left'].append(
                np.mean(d['turns_left']) if d['turns_left'] else 0)

    fig, axes = plt.subplots(1, 3, figsize=(11, 3))

    for pt in all_ranks:
        color = rank_colors.get(pt, '#333')
        label = pt.name.title()
        d = rank_data[pt]
        axes[0].plot(turns, d['alive'], 'o-', color=color, lw=2, ms=3, label=label)
        axes[1].plot(turns, d['avg_health'], 'o-', color=color, lw=2, ms=3, label=label)
        axes[2].plot(turns, d['avg_turns_left'], 'o-', color=color, lw=2, ms=3, label=label)

    axes[0].set_title('Alive by Rank', fontweight='bold', fontsize=10)
    axes[0].set_xlabel('Turn'); axes[0].set_ylim(bottom=0)

    axes[1].set_title('Avg Health by Rank', fontweight='bold', fontsize=10)
    axes[1].set_xlabel('Turn'); axes[1].set_ylim(bottom=0)

    axes[2].axhline(FOOD_RESERVE_TURNS, color='#e67e22', ls=':', lw=1,
                    label=f'reserve ({FOOD_RESERVE_TURNS})')
    axes[2].set_title('Food (turns left) by Rank', fontweight='bold', fontsize=10)
    axes[2].set_xlabel('Turn'); axes[2].set_ylim(bottom=0)

    for ax in axes:
        ax.legend(fontsize=7, loc='best')
        ax.grid(axis='y', alpha=0.3, lw=0.5)
        ax.tick_params(labelsize=8)

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Simulated {self.turn} turns — {len(self.alive)}/{len(self.pieces)} alive",
          cls="text-xs opacity-60 text-center"),
        cls="space-y-1")
```

Key changes:

- **Builds a `pinfo` lookup** from `self.pieces` to get each piece's type and diet
- **Groups all three metrics by `PieceType`** — each rank gets its own colored line
- **Consistent color scheme** — purple King, red Queen, blue Rook, green Bishop, orange Knight, grey Pawn
- **Handles missing data** — if all pieces of a rank die, their line drops to 0 gracefully
- **Reserve line** still shows on the food chart for reference

and maybe we have a chart below on the _ft_ which shows the actions as dot sizes or maybebar based upon round SimEventType. This categorical data that we want to see how it changes over time. so I am happy if you have a better visualiztion. It can be the case that per round a piece could do mulitple actions and it is okay to include it in both, we just want to see how much attacking, giving are doing over time.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def __ft__(self: FoodSimulator):
    """Line charts by piece type + activity breakdown over time."""
    if not self.snapshots:
        return P("No turns simulated yet", cls="text-sm opacity-50")

    pinfo = {p.id: (p.piece_type, p.diet) for p in self.pieces}
    all_ranks = sorted({pt for pt, _ in pinfo.values()},
                       key=lambda p: RANK_ORDER.get(p, 99))

    rank_colors = {
        PieceType.KING:   '#9b59b6', PieceType.QUEEN:  '#e74c3c',
        PieceType.ROOK:   '#3498db', PieceType.BISHOP: '#2ecc71',
        PieceType.KNIGHT: '#f39c12', PieceType.PAWN:   '#95a5a6',
    }

    turns = [s['turn'] for s in self.snapshots]
    rank_data = {pt: {'alive': [], 'avg_health': [], 'avg_turns_left': []}
                 for pt in all_ranks}

    for snap in self.snapshots:
        by_rank = {pt: {'alive': 0, 'healths': [], 'turns_left': []}
                   for pt in all_ranks}
        for pid, state in snap['piece_states'].items():
            pt, diet = pinfo.get(pid, (None, 1.0))
            if pt is None: continue
            if state['health'] > 0:
                by_rank[pt]['alive'] += 1
                by_rank[pt]['healths'].append(state['health'])
                by_rank[pt]['turns_left'].append(state['food'] / max(diet, 0.01))
        for pt in all_ranks:
            d = by_rank[pt]
            rank_data[pt]['alive'].append(d['alive'])
            rank_data[pt]['avg_health'].append(
                np.mean(d['healths']) if d['healths'] else 0)
            rank_data[pt]['avg_turns_left'].append(
                np.mean(d['turns_left']) if d['turns_left'] else 0)

    # ── Activity data — tactical events only (skip routine move/rotate/eat) ──
    tactical_events = [
        SimEventType.HARVESTED, SimEventType.GAVE,
        SimEventType.ATTACKED,  SimEventType.COUNTERED,
        SimEventType.BUMPED,    SimEventType.STARVED, SimEventType.DIED,
    ]
    event_colors = {
        SimEventType.HARVESTED: '#2ecc71', SimEventType.GAVE:      '#1abc9c',
        SimEventType.ATTACKED:  '#e74c3c', SimEventType.COUNTERED: '#c0392b',
        SimEventType.BUMPED:    '#9b59b6', SimEventType.STARVED:   '#f39c12',
        SimEventType.DIED:      '#2c3e50',
    }
    max_turn = max(turns) if turns else 0
    event_counts = {et: np.zeros(max_turn + 1) for et in tactical_events}
    ev_lookup = {et.value: et for et in tactical_events}
    for row in self._rows:
        et = ev_lookup.get(row['event'])
        if et and row['turn'] <= max_turn:
            event_counts[et][row['turn']] += 1

    active = [et for et in tactical_events if event_counts[et].any()]

    # ── Layout: 3 rank charts on top, stacked bar below ──
    fig = plt.figure(figsize=(11, 6.5))
    gs = fig.add_gridspec(2, 3, height_ratios=[1, 0.85], hspace=0.4)
    axes_top = [fig.add_subplot(gs[0, i]) for i in range(3)]
    ax_act = fig.add_subplot(gs[1, :])

    # ── Top row: rank lines ──
    for pt in all_ranks:
        c = rank_colors.get(pt, '#333'); lbl = pt.name.title()
        d = rank_data[pt]
        axes_top[0].plot(turns, d['alive'], 'o-', color=c, lw=2, ms=3, label=lbl)
        axes_top[1].plot(turns, d['avg_health'], 'o-', color=c, lw=2, ms=3, label=lbl)
        axes_top[2].plot(turns, d['avg_turns_left'], 'o-', color=c, lw=2, ms=3, label=lbl)

    axes_top[0].set_title('Alive by Rank', fontweight='bold', fontsize=10)
    axes_top[0].set_xlabel('Turn'); axes_top[0].set_ylim(bottom=0)
    axes_top[1].set_title('Avg Health by Rank', fontweight='bold', fontsize=10)
    axes_top[1].set_xlabel('Turn'); axes_top[1].set_ylim(bottom=0)
    axes_top[2].axhline(FOOD_RESERVE_TURNS, color='#e67e22', ls=':', lw=1,
                        label=f'reserve ({FOOD_RESERVE_TURNS})')
    axes_top[2].set_title('Food (turns left) by Rank', fontweight='bold', fontsize=10)
    axes_top[2].set_xlabel('Turn'); axes_top[2].set_ylim(bottom=0)
    for ax in axes_top:
        ax.legend(fontsize=7, loc='best')
        ax.grid(axis='y', alpha=0.3, lw=0.5); ax.tick_params(labelsize=8)

    # ── Bottom row: stacked bar of tactical events ──
    turn_range = np.arange(max_turn + 1)
    if active:
        bottoms = np.zeros(max_turn + 1)
        for et in active:
            vals = event_counts[et]
            ax_act.bar(turn_range, vals, bottom=bottoms, width=0.8,
                       color=event_colors.get(et, '#999'),
                       label=et.value.title(), alpha=0.85)
            bottoms += vals

    ax_act.set_title('Tactical Actions per Turn', fontweight='bold', fontsize=10)
    ax_act.set_xlabel('Turn'); ax_act.set_ylabel('Count')
    ax_act.legend(fontsize=7, loc='upper right',
                  ncol=min(len(active), 4) if active else 1)
    ax_act.grid(axis='y', alpha=0.3, lw=0.5); ax_act.tick_params(labelsize=8)
    ax_act.set_ylim(bottom=0)

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Simulated {self.turn} turns — {len(self.alive)}/{len(self.pieces)} alive",
          cls="text-xs opacity-60 text-center"),
        cls="space-y-1")
```

The bottom chart is a **stacked bar** showing only the *tactical* events per turn — I deliberately excluded the routine noise (`MOVED`, `ROTATED`, `ATE`, `BLOCKED`) since they fire every turn for every piece and would drown out the signal. What you see instead:

- 🟢 **Harvested / Gave** — economy actions (green tones)
- 🔴 **Attacked / Countered** — combat (red tones)
- 🟣 **Bumped** — friendly swaps (purple)
- 🟠 **Starved** — desperation (orange)
- ⚫ **Died** — losses (dark)

So you can read the story: early turns are all green (harvesting/sharing), then red appears when combat kicks in, orange spikes = supply crisis, black = casualties. The proportions shift as the game evolves.

did I put these in correctly?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Almost! Two issues with `_try_forward`:

**1. Missing `@patch` decorator and type annotation.** It's a standalone `def` right now, so it never gets attached to `FoodSimulator`. It needs:

```python
@patch
def _try_forward(self: FoodSimulator, piece, occupied, slot=None):
```

**2. Stray leading space.** There's a space before `def` that'll cause an `IndentationError`.

The bump logic inside it and `_try_bump` both look correct — the integration points are exactly right. Just fix the decorator and the indent on that one cell and you're good.

In [ ]:
#| export
@patch
def squad_chart(self: FoodSimulator, squad: Squad = None, figsize=(10, 4)):
    """Per-piece line chart of health & food over turns."""
    if not self.snapshots:
        print("No snapshots yet"); return

    pieces = squad.pieces if squad else self.pieces
    pid_set = {p.id for p in pieces}
    pid_names = {p.id: f"{p.name} ({p.piece_type.value})" for p in pieces}

    turns = [s['turn'] for s in self.snapshots]

    # Extract per-piece time series from existing snapshots
    health = {pid: [] for pid in pid_set}
    food   = {pid: [] for pid in pid_set}

    for snap in self.snapshots:
        states = snap.get('piece_states', {})
        for pid in pid_set:
            st = states.get(pid)
            health[pid].append(st['health'] if st else 0)
            food[pid].append(st['food']     if st else 0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    for pid in pid_set:
        lbl = pid_names.get(pid, pid[:8])
        ax1.plot(turns, health[pid], 'o-', ms=3, lw=1.5, label=lbl)
        ax2.plot(turns, food[pid],   'o-', ms=3, lw=1.5, label=lbl)

    ax1.set_title('Health per turn', fontweight='bold', fontsize=10)
    ax1.set_xlabel('Turn'); ax1.set_ylim(bottom=0)
    ax1.legend(fontsize=7, loc='lower left')
    ax1.grid(axis='y', alpha=0.3)

    ax2.set_title('Food per turn', fontweight='bold', fontsize=10)
    ax2.set_xlabel('Turn'); ax2.set_ylim(bottom=0)
    ax2.legend(fontsize=7, loc='upper right')
    ax2.grid(axis='y', alpha=0.3)

    fig.tight_layout()
    plt.show()


In [ ]:
#| export
def SquadFoodTrailOverlay(sim: 'FoodSimulator', squad: Squad = None,
                          min_opacity: float = 0.06,
                          max_opacity: float = 0.65,
                          sim_grid: HexGrid = None,
                          **kw) -> OverlaySpec:
    """Temporal food trail — pip circles sized by food %, arrows for shares, X for deaths.

    Opacity fades forward: earliest turns are faintest, most recent are boldest.
    Uses DiagramGlyphs for consistent styling with the rest of the overlay system.
    """

    def render(ctx) -> str:
        grid  = ctx.grid
        c2f   = getattr(ctx, 'c2f', None)
        board = getattr(ctx, 'board', None)

        piece_ids = ({p.id for p in squad.pieces} if squad
                     else {p.id for p in sim.pieces})

        snaps = sim.snapshots
        if not snaps:
            return ''
        max_turn = max(s['turn'] for s in snaps) or 1

        # Event log
        df = sim.df
        gives  = df[(df.event == 'gave')  & (df.piece_id.isin(piece_ids))]
        deaths = df[(df.event == 'died')  & (df.piece_id.isin(piece_ids))]

        r = grid.radius if hasattr(grid, 'radius') else 20

        # Build one DiagramGlyphs per flag we encounter, register styles once
        flag_glyphs: dict = {}
        def _get_glyphs(flag):
            key = id(flag) if flag else 'default'
            if key not in flag_glyphs:
                g = DiagramGlyphs(flag or CountryFlag.seaborn("husl", 1)[0], size=r * 0.7)
                g.register_styles(ctx.builder)
                # Register pip styles for food coloring
                for pct in range(0, 101, 5):
                    f = pct / 100
                    color = StyleCSS.lerp_color('#e74c3c', '#27ae60', f)
                    ctx.builder.add_style(StyleCSS(
                        f"pip_{pct}", fill=color, stroke=color,
                        stroke_width=0.8, opacity=0.85))
                flag_glyphs[key] = g
            return flag_glyphs[key]

        # Map piece_id → flag for glyph lookup
        piece_flags = {}
        src_pieces = squad.pieces if squad else sim.pieces
        for p in src_pieces:
            piece_flags[p.id] = p.flag

        def _render_idx(loc):
            ri = _map_point(loc, c2f) if c2f else loc
            return ri if 0 <= ri < len(grid.hexes) else -1

        parts = []

        # ── Per-turn food pips + share arrows ──
        for snap in snaps:
            t = snap['turn']
            alpha = min_opacity + (max_opacity - min_opacity) * (t / max_turn)
            states = snap.get('piece_states', {})

            # Food pips
            for pid, st in states.items():
                if pid not in piece_ids or st['health'] <= 0:
                    continue
                ri = _render_idx(st['location'])
                if ri < 0:
                    continue

                c = grid.hexes[ri].center
                cap = max(st['food_capacity'], 0.01)
                frac = np.clip(st['food'] / cap, 0, 1)

                glyphs = _get_glyphs(piece_flags.get(pid))
                pip_svg = glyphs.food_pip(MapCord(c.x, c.y), float(frac))
                parts.append(f'<g opacity="{alpha:.2f}">{pip_svg}</g>')

            # Give arrows for this turn
            turn_gives = gives[gives.turn == t]
            for _, row in turn_gives.iterrows():
                g_st = states.get(row.piece_id)
                r_st = states.get(row.dst_piece_id)
                if not g_st or not r_st:
                    continue
                gi = _render_idx(g_st['location'])
                ri2 = _render_idx(r_st['location'])
                if gi < 0 or ri2 < 0 or gi == ri2:
                    continue

                gc = grid.hexes[gi].center
                rc = grid.hexes[ri2].center
                glyphs = _get_glyphs(piece_flags.get(row.piece_id))
                arrow_svg = glyphs.transfer_arrow(
                    MapCord(gc.x, gc.y), MapCord(rc.x, rc.y))
                parts.append(f'<g opacity="{alpha:.2f}">{arrow_svg}</g>')

        # ── Death X markers (always prominent) ──
        for _, row in deaths.iterrows():
            ri = _render_idx(int(row.src_hex))
            if ri < 0:
                continue
            c = grid.hexes[ri].center
            glyphs = _get_glyphs(piece_flags.get(row.piece_id))
            parts.append(glyphs.death_marker(MapCord(c.x, c.y)))

        return '\n'.join(parts)

    return OverlaySpec("food_trail", render, priority=68)


In [ ]:
# Reset & apply food profiles (now that FOOD_RESERVE_TURNS exists)
  

# Find the queen
queen = [p for p in myStuff.queenGuard.pieces if p.piece_type == PieceType.QUEEN][0]
queen.facing = 1  # West

# Queen marches west
myStuff.queen.instructions = InstructionList(
    [Instruction.FORWARD.value] * 10, cursor=0, patrol=False)

# Pawns: harvest + give patrol (stationary food stations)
for p in myStuff.queenGuard.pieces:
    if p.piece_type == PieceType.PAWN:
        facing_to_queen = _facing_toward(myStuff.grid, p.location, myStuff.queen.location)
        p.facing = facing_to_queen
        p.instructions = InstructionList(
            [Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.HARVEST.value, Instruction.GIVE.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value],
            cursor=0, patrol=True)

        #p.instructions = []
        for i in range(0, 30):
            p.instructions.rules.append(Instruction.FORWARD.value)

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=myStuff.food_tiers,
    pieces=list(myStuff.queenGuard.pieces),
    countries=myStuff.terr.fields.get('country'),
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)
sim.run(num_turns=18)
#sim.squad_chart(myStuff.queenGuard)



myStuff.clearTerr()
region = HexRegion(hexes=set(myStuff.terr.ring(myStuff.queen.location, 7)), hexGrid=myStuff.grid)
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(scale=5),
    KingdomNamesOverlay(),
    PieceOverlay([myStuff.queenGuard]),
    #FoodOverlay(color="#8D6E63", n_tiers=6),
    SquadFoodTrailOverlay(sim, myStuff.queenGuard, sim_grid=myStuff.grid),
    #SquadPlanOverlay([queenGuard], num_turns=8),
    terrain=myStuff.terr,
    basins=myStuff.basin,
    region=region,
    debug = not showDemo
)

In [ ]:
show(myStuff.queen)

GroupedPieceList(myStuff.queenGuard.pieces,grid=myStuff.grid)

In [ ]:
show(myStuff.queenGuard)

In [ ]:
#sim.squad_chart(myStuff.queenGuard)


In [ ]:
#| export
# === Supply Line Concept Demo ===
myStuff = GameParts()  # reset


soil = SoilSystem.from_plates(myStuff.terr, [])

fy = FoodYield(myStuff.terr, myStuff.basin)
fy.compute()
#show(fy)
myStuff.fy = fy

queen = [p for p in myStuff.queenGuard.pieces if p.piece_type == PieceType.QUEEN][0]
pawns = [p for p in myStuff.queenGuard.pieces if p.piece_type == PieceType.PAWN]
queen.facing = 1  # march west

food_tiers = myStuff.fy.tiers
countries = myStuff.terr.fields.get('country')
elevs = myStuff.terr.elevations
grid = myStuff.grid

# --- Trace queen's march path (straight line, stop at impassable) ---
path = [queen.location]
loc = queen.location
for _ in range(30):
    d = HexPosition.directions()[queen.facing % 6]
    nbr = grid.hexposition_to_index(d, loc)
    if (nbr < 0 or nbr >= len(elevs) or nbr in grid.invalidRegion
        or elevs[nbr] < 1): break
    if countries is not None and countries[nbr] < 0: break
    path.append(nbr)
    loc = nbr

print(f"March path: {len(path)} hexes")

# Queen: forward + harvest every 3rd step so she grazes on the move
qrules = []
for i in range(len(path) - 1):
    qrules.append(Instruction.FORWARD.value)
    if (i + 1) % 3 == 0:
        qrules.append(Instruction.HARVEST.value)
queen.instructions = InstructionList(qrules, cursor=0, patrol=False)

# --- Teleport pawns to supply stations alongside the path ---
path_set = set(path)
spacing = max(1, (len(path) - 1) // len(pawns))

for i, pawn in enumerate(pawns):
    target_i = min(2 + i * spacing, len(path) - 1)
    target_hex = path[target_i]

    # Find best adjacent food tile NOT on the march path
    best, best_tier = None, 0
    for hp in HexPosition.origin().ring(1):
        adj = grid.hexposition_to_index(hp, target_hex)
        if (adj < 0 or adj >= len(elevs) or adj in grid.invalidRegion
            or elevs[adj] < 1 or adj in path_set): continue
        tier = int(food_tiers[adj])
        if tier > best_tier: best, best_tier = adj, tier

    if best is None: best = target_hex  # fallback

    pawn.location = best          # teleport (concept demo)
    pawn.facing = _facing_toward(grid, best, target_hex)
    pawn.instructions = InstructionList(
        [Instruction.HARVEST.value, Instruction.GIVE.value],
        cursor=0, patrol=True)
    print(f"  {pawn.name} → hex {best} (tier {best_tier}), feeding path[{target_i}]={target_hex}")

# --- Run ---
sim = FoodSimulator(
    grid=grid, elevations=elevs, food_tiers=food_tiers,
    pieces=list(myStuff.queenGuard.pieces),
    countries=countries,
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)
sim.run(num_turns=30)
sim.summary()


In [ ]:
#sim.squad_chart(myStuff.queenGuard)


## Recording

I think we want the food simulator to generate a pandas dataframe of the actions.

Lets start to add combat to the food simulator. It should be after eat and move (always better to fight if you don't have an empty stomach)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Fun! Before diving in, a couple of design questions to make sure the combat system fits your vision:

**Trigger:** Should combat be automatic (any enemy piece in adjacent hex triggers it), or instruction-based (a new `ATTACK` instruction in the piece's queue)?

**Targeting:** Does a piece attack the single nearest/weakest enemy, or all enemies in range?

And a quick note on the **damage model** — you already have `attack_strength` and `defense` on pieces. Are you thinking something like:

```
damage = attacker.attack_strength - defender.defense
```
...or something more nuanced (e.g. randomized, terrain modifier)?

I have added a temperement to piece which is how peacefull they are 100 = never attack 0 = always. If you haven't moved (ie you are paused, defend, settle) you do a saving throw for all opponents in your sight cone to see whether you attack. I do think we have some offense - defense kind of approach. If you have picked defend you get an attack back at them

```
dataclass
class Piece:
    """A game piece representing a group of units."""

    # Identity
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    owner_id: int = 0
    settlement_id: Optional[str] = None
    location: int = None
    name: str = ""
    birth_year: int = 1900

    # Core attributes
    size: int = 100
    health: int = 100
    max_health: int = 100

    # Vision
    sight: int = 3

    # Movement
    movement_range: int = 4
    current_position: int = -1

    # Facing & rules
    facing: int = 0
    instructions: InstructionList = field(default_factory=lambda: InstructionList([], 0, True))

    # Combat
    attack_strength: float = 1.0
    defense: float = 1.0
    move_strength: int = 3
    harvest_strength: float = 1.0
    temperment:int = 100

    # Type
    piece_type: PieceType = PieceType.PAWN
    squad: int = -1

    # Food
    food: float = 0.0
    diet: float = 1.0
    food_capacity: float = 10.0

    # Settlement state
    harvest_goal: Resources = None
    settle_progress: int = 0
    settle_threshold: int = 3

    # Non-serialised
    db: GeoStorage = None
    flag: CountryFlag = None

    PIECE_FIELDS = [
        'id', 'owner_id', 'settlement_id', 'location', 'name', 'birth_year',
        'size', 'health', 'max_health',
        'sight', 'movement_range', 'current_position',
        'facing', 'rules', 'cursor', 'patrol', 'squad',
        'attack_strength', 'defense', 'move_strength', 'harvest_strength',
        'temperment',
        'piece_type', '
        'food', 'diet', 'food_capacity',
        'harvest_goal', 'settle_progress', 'settle_threshold',
    ]

    def __post_init__(self):
        """Apply default stats from piece type if still at dataclass defaults."""
        defs = PIECE_DEFAULTS.get(self.piece_type)
        if defs is None:
            return
        atk, dfn, hp, diet, cap, harv, spd = defs

        # Only override if still at generic defaults
        if self.attack_strength == 1.0 and self.defense == 1.0:
            self.attack_strength  = atk
            self.defense          = dfn
            self.max_health       = hp
            self.health           = hp
            self.diet             = diet
            self.food_capacity    = cap
            self.harvest_strength = harv
            self.move_strength    = spd
            self.food             = cap  # start full

    def encode(self) -> str:
        """Encode piece as a single tab-delimited line."""
        vals = []
        for f in self.PIECE_FIELDS:
            if f == 'rules':
                v = ','.join(str(x) for x in self.instructions.rules)
            elif f == 'cursor':
                v = str(self.instructions.cursor)
            elif f == 'patrol':
                v = '1' if self.instructions.patrol else '0'
            else:
                v = getattr(self, f)
                if v is None:              v = ''
                elif isinstance(v, list):  v = ','.join(str(x) for x in v)
                elif isinstance(v, bool):  v = '1' if v else '0'
                elif isinstance(v, Enum):  v = v.value
                else:                      v = str(v)
            vals.append(str(v))
        return '\t'.join(vals)

    @classmethod
    def decode(cls, line: str) -> 'Piece':
        """Decode a tab-delimited line into a Piece."""
        parts = line.split('\t')
        kw = dict(zip(cls.PIECE_FIELDS, parts))

        # Ints
        for f in ['owner_id', 'size', 'health', 'max_health', 'sight',
                  'movement_range', 'current_position',
                  'facing', 'birth_year', 'squad','temperment'
                  'move_strength',
                  'settle_progress', 'settle_threshold']:
            kw[f] = int(kw[f]) if kw[f] else 0

        # Floats
        for f in ['food', 'diet', 'food_capacity',
                  'attack_strength', 'defense', 'harvest_strength']:
            kw[f] = float(kw[f]) if kw[f] else 0.0
        if kw['diet'] == 0.0: kw['diet'] = 1.0
        if kw['food_capacity'] == 0.0: kw['food_capacity'] = 10.0
        if kw['defense'] == 0.0: kw['defense'] = 1.0

        # Optional ints
        kw['location'] = int(kw['location']) if kw['location'] else None

        # Optional str
        kw['settlement_id'] = kw['settlement_id'] or None
        kw['name'] = kw['name'] or "Marvin"

        # Enums
        kw['piece_type'] = PieceType(kw['piece_type']) if kw['piece_type'] else PieceType.PAWN
        kw['harvest_goal'] = Resources(kw['harvest_goal']) if kw['harvest_goal'] else None

        # Build InstructionList from the three flat fields
        rules_str = kw.pop('rules', '')
        cursor = int(kw.pop('cursor', '0') or '0')
        patrol = kw.pop('patrol', '1') not in ('0', '')
        rules = [int(x) for x in rules_str.split(',') if x] if rules_str else []
        kw['instructions'] = InstructionList(rules, cursor, patrol)

        return cls(**kw)
    ```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice design — the temperament + saving throw approach is elegant. Before we get to combat logic, I spotted two bugs in the code you shared:

**1. Missing comma in `decode` — string concatenation bug:**
```python
'birth_year', 'squad','temperment'
'move_strength',          # ← missing comma above! becomes 'tempermentmove_strength'
```

**2. Stray unclosed quote in `PIECE_FIELDS`:**
```python
'piece_type', '           # ← stray quote
'food', ...
```

Worth fixing those before the combat code hits serialization issues.

---

Back to combat design — so to confirm your model:

- **Who fights:** only stationary pieces (PAUSE / DEFEND / SETTLE instruction) do the saving throw scan
- **Saving throw:** roll against `temperament` — lower temperament = more likely to attack
- **Target:** any enemy in the facing sight cone
- **Damage:** some `attack_strength` vs `defense` formula
- **Counter:** DEFEND pieces automatically hit back

One thing to nail down: does a piece pick **one** target per turn, or does it fight **all** enemies in its cone? And for the damage formula, are you thinking something like `damage = max(0, attack_strength - defense)`, or would you like a multiplier/ratio approach?

Thanks for the fix. I am going to have it be eveyone in the cone. I am giving us flamethrowers.
I guess the question is how many rounds/how lethal I want combat to be. A queen should be able to take out a bunch of pawns, but maybe it takes a few knights to get a queen?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

> ⚠️ *Note: the dialog has grown large enough that some earlier messages were trimmed from context.*

Flamethrower queen — love it. 🔥

The key balancing lever is how many turns a fight lasts. Two extremes:

- **Lethal (1-2 turns):** Combat is decisive, positioning matters enormously, one ambush can wipe a squad. Feels more like chess.
- **Attritional (5-10 turns):** Units weaken gradually, there's time to retreat or reinforce. Feels more like a war of logistics.

Given your existing food/health model, I'd lean toward **attritional** — it makes the queen's diet a strategic vulnerability mid-combat (she can't harvest while fighting), which ties nicely into your food economy.

A rough sketch of what the numbers might look like:

| Attacker | Defender | Turns to kill |
|----------|----------|---------------|
| Queen | Pawn | ~2 turns |
| Pawn | Queen | ~20 turns (alone, hopeless) |
| 3 Knights | Queen | ~5 turns |

That implies something like `damage_per_turn ≈ attack_strength / defense`, where the queen has attack ~8 and a pawn has defense ~0.8, giving ~10 damage/turn against pawn health of 80 — so about **2 turns** per pawn. Does that lethality feel right, or do you want it faster/slower?

Should we build an pyomo or other parameter turner to figure this out?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<context-warning>The size of the messages is too large to fit in the context window, so some non-pinned messages have been removed from the start of the dialog. Originally there were 80 messages; in the truncated dialog there are 56 messages.</context-warning>

Given that you already have Optuna + `FoodSimulator` working well for the food balancing, I'd skip Pyomo here and **reuse the same Optuna pattern**. The reasons:

- Combat is another black-box sim (not linearizable), same as food
- You already have the infrastructure: `run_scenario` + `balance_objective` pattern
- You can define a "victory table" for combat just like you did for food

The combat params you'd tune would be things like `attack_strength` and `defense` per piece type, and your target scenarios would encode intuitions like:

```
queen vs 1 pawn  → pawn dies in ~2 turns, queen barely scratched
queen vs 5 pawns → queen wins but takes meaningful damage
3 knights vs queen → queen loses after ~5 turns
```

Before building it though — do you want to first write the combat phase code itself, get it running in the simulator, and *then* wire up the Optuna tuner once you can actually run scenarios?

This was the older simulator. what should I put into a cell so we can execute in this notebook

```python

# AUTOGENERATED! DO NOT EDIT! File to edit: ../../nbs/game/07_combat.ipynb

"""
Combat balancing via Pyomo optimization.

Simulates fixed-position battles to tune offense/defense stats such that:
  - Rook = high defense, low offense (tank)
  - Knight = high offense, low defense (glass cannon)  
  - Queen = balanced powerhouse
  - Target: 1 Queen ≈ 1 Rook + 2 Knights

The model runs turn-based combat to equilibrium and checks who survives.
"""

from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional
import random

# Optional Pyomo import — graceful fallback for environments without it
try:
    import pyomo.environ as pyo
    HAS_PYOMO = True
except ImportError:
    pyo = None
    HAS_PYOMO = False


# ════════════════════════════════════════════════════════════════════════════
# Combat Stats — the knobs we want to balance
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class CombatStats:
    """Offense/defense profile for a piece type."""
    offense: float = 1.0   # damage dealt per attack
    defense: float = 1.0   # damage reduction multiplier (higher = tougher)
    health: float = 100.0  # starting HP
    attack_range: int = 1  # hexes (not used in abstract sim, but for reference)
    
    @property
    def effective_hp(self) -> float:
        """HP adjusted for defense: how much raw damage needed to kill."""
        return self.health * self.defense


# Default profiles — Rook tanky, Knight glasscannon, Queen strong all-round
# Tuned for 1 Queen ≈ 1 Rook + 2 Knights balance
#
# Balance results (with 15% damage variance):
#   Q vs R+2K: ~25-60% Queen wins (sensitive to exact values)
#   Q vs R:    100% Queen wins
#   Q vs 2K:   100% Queen wins  
#   R vs 2K:   0% Rook wins (Knights punish tanks)
#   R vs K:    100% Rook wins
#   Q vs 2R:   0% Queen wins
#
# Design philosophy:
#   - Rook: high defense (1.6), high HP (130), low offense (2.5) = tank
#   - Knight: high offense (7.5), low defense (0.5), low HP (55) = glass cannon
#   - Queen: balanced high stats, slightly weaker than R+2K combined
#
COMBAT_PROFILES = {
    'QUEEN':  CombatStats(offense=8.4, defense=1.29, health=121),  # powerhouse (~59% vs R+2K)
    'ROOK':   CombatStats(offense=2.5, defense=1.6, health=130),   # tanky, low offense
    'KNIGHT': CombatStats(offense=7.5, defense=0.5, health=55),    # glass cannon, high burst
    'BISHOP': CombatStats(offense=4.0, defense=1.0, health=100),
    'PAWN':   CombatStats(offense=1.5, defense=0.8, health=80),
    'KING':   CombatStats(offense=3.0, defense=1.5, health=100),
}


# ════════════════════════════════════════════════════════════════════════════
# Combat Unit — a piece in the battle
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class CombatUnit:
    """A combatant in the abstract battle simulation."""
    id: str
    team: int  # 0 or 1
    piece_type: str
    stats: CombatStats
    hp: float = field(init=False)
    
    def __post_init__(self):
        self.hp = self.stats.health
    
    @property
    def alive(self) -> bool:
        return self.hp > 0
    
    def take_damage(self, raw_damage: float) -> float:
        """Apply damage after defense. Returns actual damage taken."""
        actual = raw_damage / self.stats.defense
        self.hp = max(0, self.hp - actual)
        return actual
    
    def attack(self, target: 'CombatUnit') -> float:
        """Attack target, return damage dealt."""
        if not self.alive or not target.alive:
            return 0.0
        return target.take_damage(self.stats.offense)


# ════════════════════════════════════════════════════════════════════════════
# Battle Simulator — turn-based combat to conclusion
# ════════════════════════════════════════════════════════════════════════════

@dataclass 
class BattleResult:
    """Outcome of a battle simulation."""
    winner: Optional[int]  # 0, 1, or None (draw)
    turns: int
    team0_survivors: int
    team1_survivors: int
    team0_total_hp: float
    team1_total_hp: float
    
    @property
    def margin(self) -> float:
        """Positive = team0 won by more, negative = team1 won by more."""
        return self.team0_total_hp - self.team1_total_hp


def run_battle(
    team0: list[CombatUnit],
    team1: list[CombatUnit],
    max_turns: int = 100,
    target_priority: str = 'lowest_hp',  # or 'random', 'highest_offense'
    damage_variance: float = 0.0  # 0.2 = ±20% damage variance
) -> BattleResult:
    """
    Simulate turn-based combat between two teams.
    
    Each turn:
      1. All units on team0 attack (simultaneously)
      2. All units on team1 attack (simultaneously)
      3. Check for victory
    
    Target selection: each attacker picks a living enemy.
    damage_variance: adds randomness to damage (0.2 = ±20%)
    """
    for unit in team0 + team1:
        unit.hp = unit.stats.health  # reset HP
    
    def pick_target(enemies: list[CombatUnit]) -> Optional[CombatUnit]:
        living = [e for e in enemies if e.alive]
        if not living:
            return None
        if target_priority == 'lowest_hp':
            return min(living, key=lambda u: u.hp)
        elif target_priority == 'highest_offense':
            return max(living, key=lambda u: u.stats.offense)
        else:
            return random.choice(living)
    
    def do_attack(attacker: CombatUnit, target: CombatUnit):
        if not attacker.alive or not target.alive:
            return
        base_dmg = attacker.stats.offense
        if damage_variance > 0:
            mult = 1.0 + random.uniform(-damage_variance, damage_variance)
            base_dmg *= mult
        target.take_damage(base_dmg)
    
    for turn in range(1, max_turns + 1):
        # Team 0 attacks
        for attacker in team0:
            if not attacker.alive:
                continue
            target = pick_target(team1)
            if target:
                do_attack(attacker, target)
        
        # Team 1 attacks
        for attacker in team1:
            if not attacker.alive:
                continue
            target = pick_target(team0)
            if target:
                do_attack(attacker, target)
        
        # Check victory
        t0_alive = [u for u in team0 if u.alive]
        t1_alive = [u for u in team1 if u.alive]
        
        if not t0_alive and not t1_alive:
            return BattleResult(None, turn, 0, 0, 0.0, 0.0)
        if not t1_alive:
            return BattleResult(0, turn, len(t0_alive), 0,
                              sum(u.hp for u in t0_alive), 0.0)
        if not t0_alive:
            return BattleResult(1, turn, 0, len(t1_alive),
                              0.0, sum(u.hp for u in t1_alive))
    
    # Timeout — whoever has more HP wins
    t0_hp = sum(u.hp for u in team0 if u.alive)
    t1_hp = sum(u.hp for u in team1 if u.alive)
    t0_alive = len([u for u in team0 if u.alive])
    t1_alive = len([u for u in team1 if u.alive])
    
    if t0_hp > t1_hp:
        winner = 0
    elif t1_hp > t0_hp:
        winner = 1
    else:
        winner = None
    
    return BattleResult(winner, max_turns, t0_alive, t1_alive, t0_hp, t1_hp)


# ════════════════════════════════════════════════════════════════════════════
# Scenario Builder — create standard matchups
# ════════════════════════════════════════════════════════════════════════════

def make_unit(piece_type: str, team: int, idx: int = 0, 
              profiles: dict = None) -> CombatUnit:
    """Create a combat unit from piece type string."""
    profiles = profiles or COMBAT_PROFILES
    stats = profiles.get(piece_type.upper(), COMBAT_PROFILES['PAWN'])
    return CombatUnit(
        id=f"{piece_type}_{team}_{idx}",
        team=team,
        piece_type=piece_type.upper(),
        stats=stats
    )


def scenario_queen_vs_rook_knights(profiles: dict = None) -> tuple[list, list]:
    """
    The key balancing scenario: 1 Queen vs 1 Rook + 2 Knights.
    Target: should be roughly even (queen wins ~50% or slight edge).
    """
    profiles = profiles or COMBAT_PROFILES
    team0 = [make_unit('QUEEN', 0, 0, profiles)]
    team1 = [
        make_unit('ROOK', 1, 0, profiles),
        make_unit('KNIGHT', 1, 0, profiles),
        make_unit('KNIGHT', 1, 1, profiles),
    ]
    return team0, team1


def scenario_mirror(piece_type: str, count: int = 1, 
                    profiles: dict = None) -> tuple[list, list]:
    """Mirror match — same composition on both sides."""
    profiles = profiles or COMBAT_PROFILES
    team0 = [make_unit(piece_type, 0, i, profiles) for i in range(count)]
    team1 = [make_unit(piece_type, 1, i, profiles) for i in range(count)]
    return team0, team1


def scenario_custom(team0_spec: list[str], team1_spec: list[str],
                   profiles: dict = None) -> tuple[list, list]:
    """
    Custom scenario from piece type lists.
    
    Example: scenario_custom(['QUEEN', 'PAWN'], ['ROOK', 'ROOK', 'KNIGHT'])
    """
    profiles = profiles or COMBAT_PROFILES
    team0 = [make_unit(pt, 0, i, profiles) for i, pt in enumerate(team0_spec)]
    team1 = [make_unit(pt, 1, i, profiles) for i, pt in enumerate(team1_spec)]
    return team0, team1


# ════════════════════════════════════════════════════════════════════════════
# Batch Testing — run scenarios multiple times for stability
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class ScenarioSummary:
    """Aggregated results from running a scenario multiple times."""
    name: str
    runs: int
    team0_wins: int
    team1_wins: int
    draws: int
    avg_turns: float
    avg_margin: float  # positive = team0 advantage
    
    @property
    def team0_winrate(self) -> float:
        return self.team0_wins / self.runs if self.runs > 0 else 0.0
    
    @property
    def team1_winrate(self) -> float:
        return self.team1_wins / self.runs if self.runs > 0 else 0.0
    
    def __repr__(self):
        return (f"ScenarioSummary({self.name}: "
                f"T0={self.team0_winrate:.1%}, T1={self.team1_winrate:.1%}, "
                f"draws={self.draws}, avg_turns={self.avg_turns:.1f})")


def run_scenario_batch(
    team0_spec: list[str],
    team1_spec: list[str],
    runs: int = 100,
    profiles: dict = None,
    name: str = None,
    damage_variance: float = 0.15  # default 15% variance for interesting outcomes
) -> ScenarioSummary:
    """Run a scenario multiple times and aggregate results."""
    profiles = profiles or COMBAT_PROFILES
    name = name or f"{'+'.join(team0_spec)} vs {'+'.join(team1_spec)}"
    
    results = []
    for _ in range(runs):
        team0, team1 = scenario_custom(team0_spec, team1_spec, profiles)
        result = run_battle(team0, team1, target_priority='lowest_hp', 
                           damage_variance=damage_variance)
        results.append(result)
    
    t0_wins = sum(1 for r in results if r.winner == 0)
    t1_wins = sum(1 for r in results if r.winner == 1)
    draws = sum(1 for r in results if r.winner is None)
    avg_turns = sum(r.turns for r in results) / len(results)
    avg_margin = sum(r.margin for r in results) / len(results)
    
    return ScenarioSummary(name, runs, t0_wins, t1_wins, draws, avg_turns, avg_margin)


# ════════════════════════════════════════════════════════════════════════════
# Pyomo Balance Optimizer — find stats that achieve target win rates
# ════════════════════════════════════════════════════════════════════════════

def evaluate_balance(
    queen_offense: float, queen_defense: float, queen_health: float,
    rook_offense: float, rook_defense: float, rook_health: float,
    knight_offense: float, knight_defense: float, knight_health: float,
    runs: int = 50
) -> dict:
    """
    Evaluate a set of combat stats by running key scenarios.
    
    Returns metrics including the key "queen_vs_rk_winrate" which should be ~0.5.
    """
    profiles = {
        'QUEEN':  CombatStats(queen_offense, queen_defense, queen_health),
        'ROOK':   CombatStats(rook_offense, rook_defense, rook_health),
        'KNIGHT': CombatStats(knight_offense, knight_defense, knight_health),
        'BISHOP': COMBAT_PROFILES['BISHOP'],
        'PAWN':   COMBAT_PROFILES['PAWN'],
        'KING':   COMBAT_PROFILES['KING'],
    }
    
    # Key scenario: Queen vs Rook + 2 Knights
    qvrk = run_scenario_batch(['QUEEN'], ['ROOK', 'KNIGHT', 'KNIGHT'], 
                              runs=runs, profiles=profiles, name='Q_vs_R2K')
    
    # Sanity checks
    q_vs_r = run_scenario_batch(['QUEEN'], ['ROOK'], runs=runs//2, profiles=profiles)
    q_vs_2k = run_scenario_batch(['QUEEN'], ['KNIGHT', 'KNIGHT'], runs=runs//2, profiles=profiles)
    r_vs_2k = run_scenario_batch(['ROOK'], ['KNIGHT', 'KNIGHT'], runs=runs//2, profiles=profiles)
    
    return {
        'queen_vs_rk_winrate': qvrk.team0_winrate,
        'queen_vs_rk_margin': qvrk.avg_margin,
        'queen_vs_rook': q_vs_r.team0_winrate,
        'queen_vs_2knight': q_vs_2k.team0_winrate,
        'rook_vs_2knight': r_vs_2k.team0_winrate,
        'profiles': profiles,
    }


def balance_score(metrics: dict, target_qvrk: float = 0.5) -> float:
    """
    Score a balance configuration. Lower is better.
    
    Primary goal: queen_vs_rk_winrate ≈ target (default 50%)
    Secondary: queen should beat rook 1v1, queen should beat 2 knights
    """
    qvrk_error = abs(metrics['queen_vs_rk_winrate'] - target_qvrk)
    
    # Penalties for other imbalances
    penalties = 0.0
    
    # Queen should beat single rook (say 70%+)
    if metrics['queen_vs_rook'] < 0.7:
        penalties += (0.7 - metrics['queen_vs_rook']) * 0.5
    
    # Queen should beat 2 knights (say 60%+)
    if metrics['queen_vs_2knight'] < 0.6:
        penalties += (0.6 - metrics['queen_vs_2knight']) * 0.3
    
    # Rook should struggle vs 2 knights (glasscannons should punish tank)
    # Target: rook wins ~40%
    rook_2k_target = 0.4
    rook_2k_error = abs(metrics['rook_vs_2knight'] - rook_2k_target) * 0.2
    
    return qvrk_error + penalties + rook_2k_error


# ════════════════════════════════════════════════════════════════════════════
# Grid Search — simple parameter sweep (Pyomo overkill for discrete sim)
# ════════════════════════════════════════════════════════════════════════════

def grid_search_balance(
    queen_offense_range: tuple = (5.0, 9.0, 1.0),
    queen_defense_range: tuple = (1.0, 1.6, 0.2),
    rook_offense_range: tuple = (2.0, 5.0, 1.0),
    rook_defense_range: tuple = (1.5, 2.5, 0.25),
    knight_offense_range: tuple = (5.0, 8.0, 1.0),
    knight_defense_range: tuple = (0.5, 0.9, 0.1),
    runs_per_eval: int = 30,
    verbose: bool = True
) -> dict:
    """
    Grid search over stat combinations to find balanced values.
    
    Each range is (min, max, step).
    Returns best configuration found.
    """
    import numpy as np
    
    def frange(start, stop, step):
        return list(np.arange(start, stop + step/2, step))
    
    best_score = float('inf')
    best_config = None
    best_metrics = None
    
    q_offs = frange(*queen_offense_range)
    q_defs = frange(*queen_defense_range)
    r_offs = frange(*rook_offense_range)
    r_defs = frange(*rook_defense_range)
    k_offs = frange(*knight_offense_range)
    k_defs = frange(*knight_defense_range)
    
    total = len(q_offs) * len(q_defs) * len(r_offs) * len(r_defs) * len(k_offs) * len(k_defs)
    if verbose:
        print(f"Grid search: {total} combinations")
    
    # Fixed health values (can extend to search these too)
    q_hp, r_hp, k_hp = 120, 150, 70
    
    count = 0
    for qo in q_offs:
        for qd in q_defs:
            for ro in r_offs:
                for rd in r_defs:
                    for ko in k_offs:
                        for kd in k_defs:
                            count += 1
                            metrics = evaluate_balance(
                                qo, qd, q_hp,
                                ro, rd, r_hp,
                                ko, kd, k_hp,
                                runs=runs_per_eval
                            )
                            score = balance_score(metrics)
                            
                            if score < best_score:
                                best_score = score
                                best_config = {
                                    'queen': (qo, qd, q_hp),
                                    'rook': (ro, rd, r_hp),
                                    'knight': (ko, kd, k_hp),
                                }
                                best_metrics = metrics
                                if verbose:
                                    print(f"  [{count}/{total}] New best (score={score:.3f}): "
                                          f"Q={qo}/{qd}, R={ro}/{rd}, K={ko}/{kd} "
                                          f"→ Q_vs_R2K={metrics['queen_vs_rk_winrate']:.1%}")
    
    return {
        'score': best_score,
        'config': best_config,
        'metrics': best_metrics,
    }


# ════════════════════════════════════════════════════════════════════════════
# Convenience: Quick Test
# ════════════════════════════════════════════════════════════════════════════

def test_default_balance(runs: int = 100, verbose: bool = True) -> dict:
    """Test the default COMBAT_PROFILES and report results."""
    profiles = COMBAT_PROFILES
    
    scenarios = [
        (['QUEEN'], ['ROOK', 'KNIGHT', 'KNIGHT'], 'Q vs R+2K (target: ~50%)'),
        (['QUEEN'], ['ROOK'], 'Q vs R (Queen should win)'),
        (['QUEEN'], ['KNIGHT', 'KNIGHT'], 'Q vs 2K (Queen should win)'),
        (['ROOK'], ['KNIGHT', 'KNIGHT'], 'R vs 2K (Knights should win)'),
        (['ROOK'], ['KNIGHT'], 'R vs K (Rook should win)'),
        (['QUEEN'], ['ROOK', 'ROOK'], 'Q vs 2R (Rooks should win)'),
    ]
    
    results = {}
    for t0, t1, name in scenarios:
        summary = run_scenario_batch(t0, t1, runs=runs, profiles=profiles, name=name)
        results[name] = summary
        if verbose:
            print(f"{name}: Team0 wins {summary.team0_winrate:.1%}, "
                  f"avg {summary.avg_turns:.1f} turns")
    
    return results


# ════════════════════════════════════════════════════════════════════════════
# Main — run if executed directly
# ════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    print("=== Combat Balance Test ===\n")
    print("Default profiles:")
    for name, stats in COMBAT_PROFILES.items():
        print(f"  {name}: off={stats.offense}, def={stats.defense}, hp={stats.health}")
    print()
    
    test_default_balance(runs=100)

```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's what I'd put in a cell. It's three pieces: updated event types, the combat phase, and a re-patched `step()`:

```python
import random

# ── Event types (add ATTACKED + COUNTERED) ──
class SimEventType(Enum):
    MOVED     = "moved"
    BLOCKED   = "blocked"
    BUMPED    = "bumped"
    ROTATED   = "rotated"
    HARVESTED = "harvested"
    GAVE      = "gave"
    ATE       = "ate"
    STARVED   = "starved"
    DIED      = "died"
    ATTACKED  = "attacked"
    COUNTERED = "countered"

# ── Combat phase ──
@patch
def _combat_phase(self: FoodSimulator, moved_ids: set, defended_ids: set):
    """Stationary pieces roll temperament; attackers flame all enemies in sight cone."""
    for piece in list(self.alive):
        if piece.health <= 0:
            continue
        if piece.id in moved_ids:
            continue  # marching, not fighting

        # Saving throw — temperament 100 = pacifist, 0 = berserker
        if random.randint(0, 99) < getattr(piece, 'temperment', 100):
            continue

        # Sight cone → enemy list
        visible = set(piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=self.elevation_mult, facing_only=True))

        enemies = [p for p in self.alive
                   if p.owner_id != piece.owner_id
                   and p.health > 0
                   and p.location in visible]

        if not enemies:
            continue

        # 🔥 Flamethrower — hit everyone in the cone
        for target in enemies:
            damage = piece.attack_strength / max(target.defense, 0.01)
            target.health = max(0, target.health - damage)
            self._log(piece, SimEventType.ATTACKED,
                      dst_piece_id=target.id, amount=damage)

            # Defender counter-attack
            if target.id in defended_ids and target.health > 0:
                counter = target.attack_strength / max(piece.defense, 0.01)
                piece.health = max(0, piece.health - counter)
                self._log(target, SimEventType.COUNTERED,
                          dst_piece_id=piece.id, amount=counter)

            # Deaths
            if target.health <= 0:
                self._log(target, SimEventType.DIED)
            if piece.health <= 0:
                self._log(piece, SimEventType.DIED)
                break  # can't keep flaming if you're dead

# ── Updated step() — tracks moved/defended, calls combat after eat ──
@patch
def step(self: FoodSimulator):
    alive = self.alive
    if not alive: return
    max_slots = max(p.move_strength for p in alive)
    budgets = {p.id: float(p.move_strength) for p in alive}
    acted = set()
    moved_ids = set()
    defended_ids = set()
    occupied = self.occupied
    order = self._sort_move(alive)

    for slot in range(max_slots):
        for piece in order:
            if piece.health <= 0: continue
            if budgets[piece.id] <= 0: continue
            instr, cost = self._tick_one(piece, budgets[piece.id], occupied, slot=slot)
            if instr is not None:
                budgets[piece.id] -= cost
                acted.add(piece.id)
                if instr == Instruction.FORWARD:
                    moved_ids.add(piece.id)
                elif instr == Instruction.DEFEND:
                    defended_ids.add(piece.id)

    self._eat_phase()
    self._combat_phase(moved_ids, defended_ids)
    self._snapshot()
    self._fire_callbacks()
    self.turn += 1
```

A few design notes:

- **Bumped pieces CAN fight** — they didn't choose to move, so they're not in `moved_ids`. A pawn shoved aside by its queen might be cranky enough to swing at a nearby enemy.
- **Order matters** — pieces iterate in `self.alive` order, so whoever's first in the list shoots first. You might want `_sort_move` order here too, or reverse it (lowest rank fights first, since they're the front line).
- **Damage = `offense / defense`** — matches your old combat sim. With queen attack 8.4 vs pawn defense 0.8, that's 10.5/turn against 80 HP ≈ kills a pawn in ~8 turns. But she's hitting everyone in the cone simultaneously, so a cluster of 4 pawns takes ~42 damage/turn total.

One thing you'll want to add: **temperament defaults per piece type** in `PIECE_DEFAULTS`. Something like pawns at 80 (mostly peaceful foragers), knights at 30 (aggressive raiders), queen at 50?

Does it look like I put it in ok? should we build a demo

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looks good! The `SimEventType` with `ATTACKED`/`COUNTERED`, the `_combat_phase`, and the updated `step()` are all wired in correctly.

Let's build a quick demo — two squads facing each other, low temperament so they actually fight:

```python
# ── Combat Demo: Queen + 2 Pawns vs 3 Knights ──
myStuff = GameParts()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
grid = myStuff.grid; elevs = myStuff.terr.elevations
food_tiers = fy.tiers; countries = myStuff.terr.fields.get('country')

# Pick a central hex with some room
center = myStuff.queen.location
ring1 = [grid.hexposition_to_index(hp, center) for hp in HexPosition.origin().ring(1)]
ring2 = [grid.hexposition_to_index(hp, center) for hp in HexPosition.origin().ring(2)]
# Filter valid hexes
ring1 = [h for h in ring1 if 0 <= h < len(elevs) and elevs[h] >= 1]
ring2 = [h for h in ring2 if 0 <= h < len(elevs) and elevs[h] >= 1]

# ── Team 0: Queen + 2 Pawns (defenders) ──
queen = Piece(name="WarQueen", piece_type=PieceType.QUEEN, owner_id=0,
              location=center, facing=4, temperment=20,
              instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

p1 = Piece(name="Shield1", piece_type=PieceType.PAWN, owner_id=0,
           location=ring1[0], facing=4, temperment=40,
           instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

p2 = Piece(name="Shield2", piece_type=PieceType.PAWN, owner_id=0,
           location=ring1[1] if len(ring1) > 1 else ring1[0], facing=4, temperment=40,
           instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

# ── Team 1: 3 Knights (attackers) ──
k1 = Piece(name="Raider1", piece_type=PieceType.KNIGHT, owner_id=1,
           location=ring2[0], facing=1, temperment=10,
           instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

k2 = Piece(name="Raider2", piece_type=PieceType.KNIGHT, owner_id=1,
           location=ring2[1] if len(ring2) > 1 else ring2[0], facing=1, temperment=10,
           instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

k3 = Piece(name="Raider3", piece_type=PieceType.KNIGHT, owner_id=1,
           location=ring2[2] if len(ring2) > 2 else ring2[0], facing=1, temperment=10,
           instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

all_pieces = [queen, p1, p2, k1, k2, k3]

# ── Run ──
sim = FoodSimulator(
    grid=grid, elevations=elevs, food_tiers=food_tiers,
    pieces=all_pieces, countries=countries,
)
sim.run(num_turns=20)
sim.summary()

# ── Show combat events ──
df = sim.df
combat_df = df[df.event.isin(['attacked', 'countered', 'died'])]
print(f"\n{len(combat_df)} combat events:")
print(combat_df[['turn', 'event', 'piece_name', 'dst_piece_id', 'amount']].to_string(index=False))
```

This sets up a defensive position (queen + pawns on DEFEND with counter-attacks) vs aggressive knights (temperament 10 = almost always attacks). We should see the flamethrower cone in action, counter-attacks from defenders, and hopefully some deaths. The combat log will tell us if the lethality feels right or needs tuning.

In [ ]:
# ── Combat Demo: Queen + 2 Pawns vs 3 Knights ──
myStuff = GameParts()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
grid = myStuff.grid; elevs = myStuff.terr.elevations
food_tiers = fy.tiers; countries = myStuff.terr.fields.get('country')

# Pick a central hex with some room
center = myStuff.queen.location
ring1 = [grid.hexposition_to_index(hp, center) for hp in HexPosition.origin().ring(1)]
ring2 = [grid.hexposition_to_index(hp, center) for hp in HexPosition.origin().ring(2)]
# Filter valid hexes
ring1 = [h for h in ring1 if 0 <= h < len(elevs) and elevs[h] >= 1]
ring2 = [h for h in ring2 if 0 <= h < len(elevs) and elevs[h] >= 1]

# ── Team 0: Queen + 2 Pawns (defenders) ──
queen = Piece(name="WarQueen", piece_type=PieceType.QUEEN, owner_id=0,
              location=center, facing=4, temperment=20,
              instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

p1 = Piece(name="Shield1", piece_type=PieceType.PAWN, owner_id=0,
           location=ring1[0], facing=4, temperment=40,
           instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

p2 = Piece(name="Shield2", piece_type=PieceType.PAWN, owner_id=0,
           location=ring1[1] if len(ring1) > 1 else ring1[0], facing=4, temperment=40,
           instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

# ── Team 1: 3 Knights (attackers) ──
k1 = Piece(name="Raider1", piece_type=PieceType.KNIGHT, owner_id=1,
           location=ring2[0], facing=1, temperment=10,
           instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

k2 = Piece(name="Raider2", piece_type=PieceType.KNIGHT, owner_id=1,
           location=ring2[1] if len(ring2) > 1 else ring2[0], facing=1, temperment=10,
           instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

k3 = Piece(name="Raider3", piece_type=PieceType.KNIGHT, owner_id=1,
           location=ring2[2] if len(ring2) > 2 else ring2[0], facing=1, temperment=10,
           instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

all_pieces = [queen, p1, p2, k1, k2, k3]

for p in all_pieces:
    p.food = 999
    p.food_capacity = 999

# ── Run ──
sim = FoodSimulator(
    grid=grid, elevations=elevs, food_tiers=food_tiers,
    pieces=all_pieces, countries=countries,
)
sim.run(num_turns=20)
sim.summary()

# ── Show combat events ──
df = sim.df
combat_df = df[df.event.isin(['attacked', 'countered', 'died'])]
print(f"\n{len(combat_df)} combat events:")
print(combat_df[['turn', 'event', 'piece_name', 'dst_piece_id', 'amount']].to_string(index=False))


In [ ]:
show(k1)

In [ ]:
show(queen)

So I think the queen starves

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yep! Look at the timeline:

- **Turn 0-1:** Queen flames all 3 knights dead. Combat victory. 🎉
- **Turn 2:** Queen dies. Not from combat — from her 27.9 diet with no supply chain. 💀
- **Turn 7:** Pawns starve too (on DEFEND, not harvesting).

Your core design constraint is working perfectly — **the queen can't self-sustain even when she wins the fight**. But it muddies the combat demo.

Quick fix — give everyone huge food so we can see combat in isolation:

```python
for p in all_pieces:
    p.food = 999
    p.food_capacity = 999
```

Add that right before creating the `FoodSimulator`. That way starvation is off the table and you can see how many turns combat *actually* takes, whether the damage numbers feel right, etc.

The other observation from the log: the queen deals 16.8 damage/turn to each knight (8.4 / 0.5 defense = 16.8), and knights only have 55 HP, so they die in ~3.3 hits. With the flamethrower hitting all 3 simultaneously, the queen wipes the floor. Meanwhile 3 knights deal 5.8 each to the queen (7.5 / 1.29 defense) = ~17.4 total/turn against 121 HP — that's ~7 turns to kill her.

So 1 queen vs 3 knights: queen wins in ~4 turns with ~50 HP left. That feels about right for a powerhouse, but you can tweak lethality once starvation isn't confounding things.

So I wonder about placing a knight in the queens blind spot

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great instinct — that's exactly the kind of tactical play facing should create. Let's test it:

```python
# ── Blind Spot Demo: Knight behind the Queen ──
center = myStuff.queen.location
ring1 = [grid.hexposition_to_index(hp, center) for hp in HexPosition.origin().ring(1)]
ring1 = [h for h in ring1 if 0 <= h < len(elevs) and elevs[h] >= 1]

# Queen faces direction 4 (roughly east)
queen2 = Piece(name="BlindQueen", piece_type=PieceType.QUEEN, owner_id=0,
               location=center, facing=4, temperment=20,
               instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

# Knight placed BEHIND the queen — direction 1 is opposite of 4
behind_hex = grid.hexposition_to_index(HexPosition.directions()[1], center)
sneaky = Piece(name="Sneaky", piece_type=PieceType.KNIGHT, owner_id=1,
               location=behind_hex, facing=4, temperment=0,  # always attacks
               instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

# Check: is the knight in the queen's sight cone?
queen_vis = set(queen2.hexes_in_sight(grid, elevs, elevation_mult=0.005, facing_only=True))
knight_vis = set(sneaky.hexes_in_sight(grid, elevs, elevation_mult=0.005, facing_only=True))
print(f"Knight at {behind_hex} in queen's cone? {behind_hex in queen_vis}")
print(f"Queen at {center} in knight's cone? {center in knight_vis}")

for p in [queen2, sneaky]:
    p.food = 999; p.food_capacity = 999

sim2 = FoodSimulator(grid=grid, elevations=elevs, food_tiers=food_tiers,
                     pieces=[queen2, sneaky], countries=countries)
sim2.run(num_turns=30)
sim2.summary()

df2 = sim2.df
combat2 = df2[df2.event.isin(['attacked', 'countered', 'died'])]
print(f"\n{len(combat2)} combat events:")
print(combat2[['turn', 'event', 'piece_name', 'dst_piece_id', 'amount']].to_string(index=False))
```

The key questions this answers:

1. **Can the knight see the queen?** (should be yes — knight faces toward her)
2. **Can the queen see the knight?** (should be NO — knight is behind her)
3. **Does the queen's counter-attack still fire?** — right now the counter in `_combat_phase` doesn't check sight cone, it's automatic for defenders. That's a design choice: should a counter-attack require you to *see* your attacker?

If the counter fires regardless, a blind-spot knight still takes damage — just no flamethrower. If you want true blind-spot exploitation, you'd add a sight check to the counter too:

```python
# Only counter if you can see who hit you
if target.id in defended_ids and target.health > 0:
    target_vis = set(target.hexes_in_sight(
        self.grid, self.elevations,
        elevation_mult=self.elevation_mult, facing_only=True))
    if piece.location in target_vis:  # ← can you see your attacker?
        counter = target.attack_strength / max(piece.defense, 0.01)
        ...
```

That would make flanking *devastating* — and make the knight's scout role really matter for finding blind spots.

In [ ]:
# ── Blind Spot Demo: Knight behind the Queen ──
center = myStuff.queen.location
ring1 = [grid.hexposition_to_index(hp, center) for hp in HexPosition.origin().ring(1)]
ring1 = [h for h in ring1 if 0 <= h < len(elevs) and elevs[h] >= 1]

# Queen faces direction 4 (roughly east)
queen2 = Piece(name="BlindQueen", piece_type=PieceType.QUEEN, owner_id=0,
               location=center, facing=4, temperment=20,
               instructions=InstructionList([Instruction.DEFEND.value], patrol=True))

# Knight placed BEHIND the queen — direction 1 is opposite of 4
behind_hex = grid.hexposition_to_index(HexPosition.directions()[1], center)
sneaky = Piece(name="Sneaky", piece_type=PieceType.KNIGHT, owner_id=1,
               location=behind_hex, facing=4, temperment=0,  # always attacks
               instructions=InstructionList([Instruction.PAUSE.value], patrol=True))

# Check: is the knight in the queen's sight cone?
queen_vis = set(queen2.hexes_in_sight(grid, elevs, elevation_mult=0.005, facing_only=True))
knight_vis = set(sneaky.hexes_in_sight(grid, elevs, elevation_mult=0.005, facing_only=True))
print(f"Knight at {behind_hex} in queen's cone? {behind_hex in queen_vis}")
print(f"Queen at {center} in knight's cone? {center in knight_vis}")

for p in [queen2, sneaky]:
    p.food = 999; p.food_capacity = 999

sim2 = FoodSimulator(grid=grid, elevations=elevs, food_tiers=food_tiers,
                     pieces=[queen2, sneaky], countries=countries)
sim2.run(num_turns=30)
sim2.summary()

df2 = sim2.df
combat2 = df2[df2.event.isin(['attacked', 'countered', 'died'])]
print(f"\n{len(combat2)} combat events:")
print(combat2[['turn', 'event', 'piece_name', 'dst_piece_id', 'amount']].to_string(index=False))


So what would combat phase look like if you can only counter people in your vision

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Just one extra sight check in the counter block:

```python
@patch
def _combat_phase(self: FoodSimulator, moved_ids: set, defended_ids: set):
    """Stationary pieces roll temperament; attackers flame all enemies in sight cone."""
    for piece in list(self.alive):
        if piece.health <= 0:
            continue
        if piece.id in moved_ids:
            continue

        if random.randint(0, 99) < getattr(piece, 'temperment', 100):
            continue

        visible = set(piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=self.elevation_mult, facing_only=True))

        enemies = [p for p in self.alive
                   if p.owner_id != piece.owner_id
                   and p.health > 0
                   and p.location in visible]

        if not enemies:
            continue

        for target in enemies:
            damage = piece.attack_strength / max(target.defense, 0.01)
            target.health = max(0, target.health - damage)
            self._log(piece, SimEventType.ATTACKED,
                      dst_piece_id=target.id, amount=damage)

            # Counter-attack: only if defending AND can see attacker
            if target.id in defended_ids and target.health > 0:
                target_vis = set(target.hexes_in_sight(
                    self.grid, self.elevations,
                    elevation_mult=self.elevation_mult, facing_only=True))
                if piece.location in target_vis:
                    counter = target.attack_strength / max(piece.defense, 0.01)
                    piece.health = max(0, piece.health - counter)
                    self._log(target, SimEventType.COUNTERED,
                              dst_piece_id=piece.id, amount=counter)

            if target.health <= 0:
                self._log(target, SimEventType.DIED)
            if piece.health <= 0:
                self._log(piece, SimEventType.DIED)
                break
```

The only change is the 4 lines around `target_vis` — the defender computes their own sight cone and only counters if the attacker's hex is in it.

With this, the sneaky knight behind the queen should deal ~5.8 damage/turn with **zero** counter-damage. At 121 HP that's ~21 turns to kill the queen from behind. Not fast, but free damage. Two knights flanking? ~10 turns, still free. That makes scouting (knight's high sight stat) genuinely valuable — find the blind spot, position there, and the queen's flamethrower never fires.